# AI-Assisted Diagnosis of Cutaneous Melanoma

## ResUNet segmentation → lesion ROI → EfficientNetB0 classification → clinical evaluation → Grad-CAM → Gradio

This notebook is Pavani's complete ResUNet branch. It trains **only ResUNet** for lesion segmentation; Standard U-Net, Attention U-Net and U-Net++ are intentionally not trained here. The segmentation settings are kept suitable for fair comparison with those separate team experiments.

The data flow is:

**ISIC 2018 Task 1 image + expert mask → ResUNet → raw binary lesion mask → segmentation metrics → SIIM-ISIC 2020 image → ResUNet mask → padded lesion ROI → EfficientNetB0 → model probability of malignancy → clinical metrics → Grad-CAM → Gradio**

> **Research/educational prototype only. This system is not a medical diagnosis and must not replace assessment by a qualified healthcare professional.**

No performance values are pre-filled. All result tables and figures are produced only after the notebook is run on the stated data.


## Execution guide

1. In Colab choose **Runtime → Change runtime type → T4 GPU** (or another GPU).
2. **Section 2** contains all experiment settings. **Section 3** mounts Drive and controls persistent storage.
3. To use the team's shared segmentation split, upload `segmentation_split.csv` to `/content/` before running **Section 6**, or place it in the persistent output folder. The notebook uses it exactly when valid; otherwise it creates the deterministic seed-42 split.
4. **Section 15** trains ResUNet. The best checkpoint is `/content/drive/MyDrive/Melanoma_AI_Project/ResUNet_Pipeline/resunet_best.keras`.
5. **Section 22** begins the SIIM-ISIC 2020 classification stage. Large images, ZIPs and ROI crops remain under `/content/`, not Drive.
6. **Sections 29–30** train and fine-tune EfficientNetB0. **Sections 33–35** contain final metrics, the confusion matrix, ROC and precision–recall curves.
7. **Section 40** constructs and launches the Gradio application with `demo.launch(share=True)`.

The official image archives are large and full training takes hours. Colab temporary storage is erased when the runtime disconnects, while the models, split files, metrics and figures listed in Section 41 remain in Drive.


## 1. Project Overview


The segmentation experiment uses the 2,594 labelled training image–mask pairs from **ISIC 2018 Task 1**. A held-out test split is never used for training, early stopping, threshold selection or architecture selection. ResUNet is evaluated with Dice, IoU, pixel sensitivity and pixel specificity.

After ResUNet is locked, it predicts lesion masks for the labelled **SIIM-ISIC 2020** training images. Padded ROIs are generated once and written to temporary Colab storage. EfficientNetB0 then learns benign-versus-malignant classification from those ROI images using patient-disjoint splits and class weights calculated from the training split only.


## 2. Configuration


In [ ]:
# Colab already provides TensorFlow. Install only the additional packages used here.
import importlib.util
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "gradio>=5,<7", "scipy>=1.10", "scikit-learn>=1.3", "tqdm>=4.66"],
        check=True,
    )
print("Running in Google Colab:", IN_COLAB)


In [ ]:
SEED = 42

# Fair team segmentation settings
SEG_IMAGE_SIZE = (256, 256)
SEG_BATCH_SIZE = 8
SEG_EPOCHS = 50
SEG_LEARNING_RATE = 1e-3
SEG_THRESHOLD = 0.50

# Classification settings
CLS_IMAGE_SIZE = (224, 224)
CLS_BATCH_SIZE = 32
CLS_HEAD_EPOCHS = 10
CLS_FINE_TUNE_EPOCHS = 20
CLS_HEAD_LEARNING_RATE = 1e-3
CLS_FINE_TUNE_LEARNING_RATE = 1e-5
CLS_FINE_TUNE_LAST_N = 40
DEFAULT_CLS_THRESHOLD = 0.50

ROI_PADDING_FRACTION = 0.10
MIN_ROI_SIDE_PIXELS = 8
MIN_VALID_MASK_FRACTION = 0.001
MAX_VALID_MASK_FRACTION = 0.98
ROI_POSTPROCESS = False  # ROI-only option; raw masks always drive team-comparison metrics.

USE_MIXED_PRECISION = True
VERIFY_IMAGE_CONTENT = True
RUN_FINE_TUNING = True
RUN_CLASSIFIER_BASELINE = False
OPTIMISE_THRESHOLD_ON_VALIDATION = True
LAUNCH_GRADIO = True
DELETE_RUNTIME_ARCHIVES = True

EXPECTED_SEGMENTATION_CASES = 2594
EXPECTED_CLASSIFICATION_IMAGES = 33126

# Shared team split search order. Change this only if your CSV has another location.
SHARED_SEGMENTATION_SPLIT_PATH = "/content/segmentation_split.csv"


In [ ]:
import gc
import json
import math
import os
import random
import re
import shutil
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import tensorflow as tf
from PIL import Image, UnidentifiedImageError
from scipy import ndimage
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras
from tensorflow.keras import layers
from tqdm.auto import tqdm

try:
    from IPython.display import display
except ImportError:
    display = print

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic operations enabled.")
except Exception as exc:
    print("Full deterministic execution was unavailable:", exc)

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPU devices:", gpus)

MIXED_PRECISION_ENABLED = False
if gpus and USE_MIXED_PRECISION:
    try:
        keras.mixed_precision.set_global_policy("mixed_float16")
        MIXED_PRECISION_ENABLED = True
    except Exception as exc:
        print("Mixed precision was not enabled:", exc)
print("Numeric policy:", keras.mixed_precision.global_policy())


## 3. Storage


In [ ]:
# Persistent outputs only: models, split CSVs, results, figures and configuration.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/Melanoma_AI_Project/ResUNet_Pipeline")
    RUNTIME_ROOT = Path("/content/melanoma_resunet_pipeline")
    ARCHIVE_DIR = Path("/content/melanoma_resunet_downloads")
else:
    OUTPUT_DIR = Path(os.environ.get("MELANOMA_OUTPUT_DIR", "./ResUNet_Pipeline_outputs")).resolve()
    RUNTIME_ROOT = Path(os.environ.get("MELANOMA_RUNTIME_ROOT", "./runtime_data")).resolve()
    ARCHIVE_DIR = RUNTIME_ROOT / "downloads"

FIGURES_DIR = OUTPUT_DIR / "figures"
SEG_WORK_DIR = RUNTIME_ROOT / "isic2018_task1"
CLS_WORK_DIR = RUNTIME_ROOT / "siim_isic2020"
ROI_DIR = RUNTIME_ROOT / "classification_rois"

for folder in [OUTPUT_DIR, FIGURES_DIR, RUNTIME_ROOT, ARCHIVE_DIR, SEG_WORK_DIR, CLS_WORK_DIR, ROI_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "seed": SEED,
    "segmentation_architecture": "ResUNet only",
    "segmentation_image_size": list(SEG_IMAGE_SIZE),
    "segmentation_batch_size": SEG_BATCH_SIZE,
    "segmentation_max_epochs": SEG_EPOCHS,
    "segmentation_learning_rate": SEG_LEARNING_RATE,
    "segmentation_threshold": SEG_THRESHOLD,
    "segmentation_loss": "binary_crossentropy_plus_dice_loss",
    "classification_image_size": list(CLS_IMAGE_SIZE),
    "classification_batch_size": CLS_BATCH_SIZE,
    "classification_head_epochs": CLS_HEAD_EPOCHS,
    "classification_fine_tune_epochs": CLS_FINE_TUNE_EPOCHS,
    "classification_head_learning_rate": CLS_HEAD_LEARNING_RATE,
    "classification_fine_tune_learning_rate": CLS_FINE_TUNE_LEARNING_RATE,
    "roi_padding_fraction": ROI_PADDING_FRACTION,
    "roi_postprocess": ROI_POSTPROCESS,
    "dataset_storage": "temporary_colab_content",
    "persistent_output_directory": str(OUTPUT_DIR),
}
(OUTPUT_DIR / "resunet_pipeline_config.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")

print("Persistent outputs:", OUTPUT_DIR)
print("Temporary datasets and ROIs:", RUNTIME_ROOT)
print("Large datasets will not be copied to Google Drive.")


## 4. ISIC 2018 Download


The direct files below are linked from the official ISIC Challenge data archive. Each ZIP is downloaded into `/content`, integrity-checked, safely extracted, and deleted after successful extraction. Existing extracted data in the current runtime are reused.


In [ ]:
# ============================================================
# OFFICIAL DATASET FILES + RUNTIME DOWNLOAD HELPERS
# ============================================================
# Important:
# This cell only DEFINES the files and download functions.
# It does NOT immediately download every large archive.
#
# The next extraction cell downloads ONE large ZIP at a time
# directly into /content, extracts it, and then deletes the ZIP
# (when DELETE_RUNTIME_ARCHIVES=True). This avoids using Drive
# quota and reduces peak runtime storage.

DATASET_SPECS = {
    "seg_images": {
        "filename": "ISIC2018_Task1-2_Training_Input.zip",
        "url": "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task1-2_Training_Input.zip",
        "kind": "zip",
    },
    "seg_masks": {
        "filename": "ISIC2018_Task1_Training_GroundTruth.zip",
        "url": "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task1_Training_GroundTruth.zip",
        "kind": "zip",
    },
    "cls_images": {
        "filename": "ISIC_2020_Training_JPEG.zip",
        "url": "https://isic-archive.s3.amazonaws.com/challenges/2020/ISIC_2020_Training_JPEG.zip",
        "kind": "zip",
    },
    "cls_ground_truth": {
        "filename": "ISIC_2020_Training_GroundTruth.csv",
        "url": "https://isic-archive.s3.amazonaws.com/challenges/2020/ISIC_2020_Training_GroundTruth.csv",
        "kind": "csv",
    },
    "cls_metadata_v2": {
        "filename": "ISIC_2020_Training_GroundTruth_v2.csv",
        "url": "https://isic-archive.s3.amazonaws.com/challenges/2020/ISIC_2020_Training_GroundTruth_v2.csv",
        "kind": "csv",
    },
    "cls_duplicates": {
        "filename": "ISIC_2020_Training_Duplicates.csv",
        "url": "https://isic-archive.s3.amazonaws.com/challenges/2020/ISIC_2020_Training_Duplicates.csv",
        "kind": "csv",
    },
}


def file_is_valid(path, kind):
    path = Path(path)
    if not path.exists() or path.stat().st_size < 100:
        return False

    if kind == "zip":
        return zipfile.is_zipfile(path)

    if kind == "csv":
        try:
            return len(pd.read_csv(path, nrows=3).columns) >= 1
        except Exception:
            return False

    return True


def remote_file_size(url, timeout=30):
    try:
        response = requests.head(
            url,
            allow_redirects=True,
            timeout=timeout,
        )
        response.raise_for_status()
        value = response.headers.get("Content-Length")
        return int(value) if value else None
    except Exception:
        return None


def download_with_resume(spec, destination_dir=None, chunk_mb=8):
    """
    Download one official ISIC file into temporary runtime storage.

    Large data are intentionally NOT saved to Google Drive.
    """
    destination_dir = Path(destination_dir or ARCHIVE_DIR)
    destination_dir.mkdir(parents=True, exist_ok=True)

    final_path = destination_dir / spec["filename"]
    part_path = final_path.with_suffix(final_path.suffix + ".part")

    if file_is_valid(final_path, spec["kind"]):
        print("Valid runtime file already exists; skipping:", final_path.name)
        return final_path

    remote_size = remote_file_size(spec["url"])
    start_byte = part_path.stat().st_size if part_path.exists() else 0
    headers = {"Range": f"bytes={start_byte}-"} if start_byte else {}
    mode = "ab" if start_byte else "wb"

    try:
        with requests.get(
            spec["url"],
            stream=True,
            headers=headers,
            timeout=(30, 120),
        ) as response:
            response.raise_for_status()

            if start_byte and response.status_code != 206:
                print("Server did not honour resume; restarting this file.")
                start_byte = 0
                mode = "wb"

            total = remote_size or int(
                response.headers.get("Content-Length", 0)
            ) or None

            with open(part_path, mode) as handle, tqdm(
                total=total,
                initial=start_byte,
                unit="B",
                unit_scale=True,
                desc=spec["filename"],
            ) as progress:
                for chunk in response.iter_content(
                    chunk_size=chunk_mb * 1024 * 1024
                ):
                    if chunk:
                        handle.write(chunk)
                        progress.update(len(chunk))

        part_path.replace(final_path)

        if not file_is_valid(final_path, spec["kind"]):
            raise RuntimeError(
                "Downloaded file failed its integrity check."
            )

        return final_path

    except Exception as exc:
        message = (
            f"Automatic download failed for {spec['filename']}.\n"
            f"The file was being downloaded to temporary Colab storage:\n"
            f"{final_path}\n\n"
            "You do NOT need to place the large archive in Google Drive.\n"
            f"Original error: {exc}"
        )
        print(message)
        raise RuntimeError(message) from exc


def runtime_extraction_complete(destination):
    return (Path(destination) / ".extraction_complete").exists()


print("Dataset download helpers ready.")
print("No large dataset has been written to Google Drive by this cell.")


In [ ]:
# ============================================================
# DOWNLOAD TO /content -> EXTRACT -> DELETE ZIP
# ============================================================
# This is intentionally sequential:
#
# 1. Download one ZIP to temporary Colab storage.
# 2. Extract it into /content.
# 3. Delete the ZIP immediately after successful extraction.
# 4. Move to the next ZIP.
#
# tf.data later reads the extracted JPEG/PNG files lazily in
# batches, so decoded images are not all held in RAM.


def zip_uncompressed_size(zip_path):
    with zipfile.ZipFile(zip_path) as archive:
        return sum(info.file_size for info in archive.infolist())


def available_bytes(path):
    return shutil.disk_usage(Path(path)).free


def safe_extract_zip(
    zip_path,
    destination,
    marker_name=".extraction_complete",
):
    zip_path = Path(zip_path)
    destination = Path(destination)
    marker = destination / marker_name

    if marker.exists():
        print("Extraction marker found; skipping:", destination)
        return destination

    required = int(zip_uncompressed_size(zip_path) * 1.15)
    free = available_bytes(destination.parent)

    print(
        f"{zip_path.name}: approximately "
        f"{required / 1e9:.1f} GB extraction space required; "
        f"{free / 1e9:.1f} GB currently free."
    )

    if free < required:
        raise RuntimeError(
            f"Not enough temporary Colab disk space to extract "
            f"{zip_path.name}. Need roughly "
            f"{(required - free) / 1e9:.1f} GB more free space."
        )

    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()

    with zipfile.ZipFile(zip_path) as archive:
        for member in tqdm(
            archive.infolist(),
            desc=f"Extracting {zip_path.name}",
        ):
            target = (destination / member.filename).resolve()

            if root not in target.parents and target != root:
                raise RuntimeError(
                    f"Unsafe ZIP member path: {member.filename}"
                )

            archive.extract(member, destination)

    marker.write_text(
        f"Extracted from {zip_path.name}\n",
        encoding="utf-8",
    )

    return destination


def download_extract_and_cleanup(
    dataset_key,
    destination,
):
    """
    Download and extract one large ZIP only when its extracted
    dataset is not already available in the current runtime.
    """
    destination = Path(destination)

    if runtime_extraction_complete(destination):
        print(
            f"{dataset_key}: extracted data already exists in "
            f"the current runtime; no download required."
        )
        return None

    archive_path = download_with_resume(
        DATASET_SPECS[dataset_key],
        destination_dir=ARCHIVE_DIR,
    )

    safe_extract_zip(
        archive_path,
        destination,
    )

    if (
        DELETE_RUNTIME_ARCHIVES
        and Path(archive_path).exists()
    ):
        Path(archive_path).unlink()
        print(
            "Deleted temporary ZIP after successful extraction:",
            Path(archive_path).name,
        )

    return archive_path




download_extract_and_cleanup("seg_images", SEG_WORK_DIR / "images")
download_extract_and_cleanup("seg_masks", SEG_WORK_DIR / "masks")
print("ISIC 2018 Task 1 files are ready under:", SEG_WORK_DIR)


## 5. Segmentation Manifest


In [ ]:
def image_id_from_image_path(path):
    return Path(path).stem


def image_id_from_mask_path(path):
    return re.sub(r"_segmentation$", "", Path(path).stem, flags=re.IGNORECASE)


seg_image_paths = [
    path for path in (SEG_WORK_DIR / "images").rglob("*")
    if path.suffix.lower() in {".jpg", ".jpeg"}
]
seg_mask_paths = [
    path for path in (SEG_WORK_DIR / "masks").rglob("*")
    if path.suffix.lower() == ".png"
]

images_table = pd.DataFrame({
    "image_id": [image_id_from_image_path(path) for path in seg_image_paths],
    "image_path": [str(path) for path in seg_image_paths],
})
masks_table = pd.DataFrame({
    "image_id": [image_id_from_mask_path(path) for path in seg_mask_paths],
    "mask_path": [str(path) for path in seg_mask_paths],
})

if images_table.image_id.duplicated().any():
    raise RuntimeError("Duplicate ISIC image IDs found: " + str(images_table.loc[images_table.image_id.duplicated(False), "image_id"].head().tolist()))
if masks_table.image_id.duplicated().any():
    raise RuntimeError("Duplicate ISIC mask IDs found: " + str(masks_table.loc[masks_table.image_id.duplicated(False), "image_id"].head().tolist()))

image_ids = set(images_table.image_id)
mask_ids = set(masks_table.image_id)
if image_ids != mask_ids:
    raise RuntimeError(
        "Image-mask matching failed. IDs without masks: "
        f"{sorted(image_ids - mask_ids)[:10]}; IDs without images: {sorted(mask_ids - image_ids)[:10]}"
    )

seg_manifest = images_table.merge(masks_table, on="image_id", how="inner", validate="one_to_one")
required_seg_columns = ["image_id", "image_path", "mask_path"]
if seg_manifest[required_seg_columns].isna().any().any():
    raise RuntimeError("NaNs were found in required segmentation manifest columns.")
if not seg_manifest.image_path.map(lambda value: Path(value).exists()).all():
    raise RuntimeError("At least one segmentation image path does not exist.")
if not seg_manifest.mask_path.map(lambda value: Path(value).exists()).all():
    raise RuntimeError("At least one segmentation mask path does not exist.")
if len(seg_manifest) != EXPECTED_SEGMENTATION_CASES:
    raise RuntimeError(f"Expected {EXPECTED_SEGMENTATION_CASES} matched cases; found {len(seg_manifest)}.")

seg_manifest = seg_manifest.sort_values("image_id").reset_index(drop=True)
seg_manifest.to_csv(OUTPUT_DIR / "segmentation_manifest.csv", index=False)
print("Verified image count:", len(images_table))
print("Verified mask count:", len(masks_table))
display(seg_manifest.head())


## 6. Team Segmentation Split


In [ ]:
# ============================================================
# USE ONLY 100 ISIC 2018 IMAGE-MASK PAIRS
# ============================================================

# Randomly select exactly 100 cases reproducibly
seg_manifest = (
    seg_manifest
    .sample(n=100, random_state=SEED)
    .sort_values("image_id")
    .reset_index(drop=True)
)

print("Using only:", len(seg_manifest), "image-mask pairs")

# 70% training, 15% validation, 15% test
train_ids, temporary_ids = train_test_split(
    seg_manifest["image_id"].to_numpy(),
    test_size=0.30,
    random_state=SEED,
    shuffle=True
)

validation_ids, test_ids = train_test_split(
    temporary_ids,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

segmentation_split = pd.DataFrame({
    "image_id": np.concatenate([
        train_ids,
        validation_ids,
        test_ids
    ]),
    "split": (
        ["train"] * len(train_ids)
        + ["validation"] * len(validation_ids)
        + ["test"] * len(test_ids)
    )
})

# Save the split
segmentation_split = segmentation_split.sort_values(
    ["split", "image_id"]
).reset_index(drop=True)

segmentation_split.to_csv(
    OUTPUT_DIR / "segmentation_split.csv",
    index=False
)

# Join split information back to the 100 selected cases
seg_manifest_with_split = seg_manifest.merge(
    segmentation_split,
    on="image_id",
    validate="one_to_one"
)

train_seg_df = (
    seg_manifest_with_split
    .query("split == 'train'")
    .reset_index(drop=True)
)

val_seg_df = (
    seg_manifest_with_split
    .query("split == 'validation'")
    .reset_index(drop=True)
)

test_seg_df = (
    seg_manifest_with_split
    .query("split == 'test'")
    .reset_index(drop=True)
)

# Check that no image appears in more than one split
train_set = set(train_seg_df.image_id)
val_set = set(val_seg_df.image_id)
test_set = set(test_seg_df.image_id)

assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)

print("Training:", len(train_seg_df))
print("Validation:", len(val_seg_df))
print("Test:", len(test_seg_df))
print("Total:", len(seg_manifest))
print("ZERO image-ID overlap confirmed.")

## 7. Data Validation


In [ ]:
def load_rgb_pil(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def load_mask_pil(path):
    with Image.open(path) as mask:
        return mask.convert("L").copy()


def validate_segmentation_files(frame):
    failures = []
    for row in tqdm(frame.itertuples(index=False), total=len(frame), desc="Checking ISIC image-mask files"):
        try:
            image = load_rgb_pil(row.image_path)
            mask = load_mask_pil(row.mask_path)
            if image.size != mask.size:
                failures.append({"image_id": row.image_id, "problem": f"dimension mismatch {image.size} vs {mask.size}"})
            mask_values = np.asarray(mask)
            if mask_values.size == 0:
                failures.append({"image_id": row.image_id, "problem": "empty mask file"})
        except Exception as exc:
            failures.append({"image_id": row.image_id, "problem": repr(exc)})
    return pd.DataFrame(failures, columns=["image_id", "problem"])


seg_file_failures = validate_segmentation_files(seg_manifest)
seg_file_failures.to_csv(OUTPUT_DIR / "segmentation_file_validation_failures.csv", index=False)
if len(seg_file_failures):
    raise RuntimeError("Segmentation file validation failed. Inspect segmentation_file_validation_failures.csv")
print("All segmentation image and mask files opened successfully with matching dimensions.")


## 8. Exploratory Visualisation


In [ ]:
sample_rows = train_seg_df.sample(n=min(3, len(train_seg_df)), random_state=SEED)
fig, axes = plt.subplots(len(sample_rows), 3, figsize=(13, 4 * len(sample_rows)))
axes = np.atleast_2d(axes)
for row_index, (_, row) in enumerate(sample_rows.iterrows()):
    image = load_rgb_pil(row.image_path)
    mask = load_mask_pil(row.mask_path)
    axes[row_index, 0].imshow(image)
    axes[row_index, 0].set_title(f"Original image\n{row.image_id}")
    axes[row_index, 1].imshow(mask, cmap="gray")
    axes[row_index, 1].set_title("Expert ground-truth mask")
    axes[row_index, 2].imshow(image)
    axes[row_index, 2].imshow(mask.resize(image.size, Image.Resampling.NEAREST), cmap="Reds", alpha=0.35)
    axes[row_index, 2].set_title("Ground-truth overlay")
    for axis in axes[row_index]:
        axis.axis("off")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "segmentation_data_sanity.png", dpi=180, bbox_inches="tight")
plt.show()


## 9. tf.data Segmentation Pipeline


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def decode_segmentation_pair(image_path, mask_path):
    image = tf.io.decode_jpeg(tf.io.read_file(image_path), channels=3)
    image = tf.image.resize(image, SEG_IMAGE_SIZE, method="bilinear")
    image = tf.cast(image, tf.float32) / 255.0
    image.set_shape((*SEG_IMAGE_SIZE, 3))

    mask = tf.io.decode_png(tf.io.read_file(mask_path), channels=1)
    mask = tf.image.resize(mask, SEG_IMAGE_SIZE, method="nearest")
    mask = tf.cast(mask > 127, tf.float32)
    mask.set_shape((*SEG_IMAGE_SIZE, 1))
    return image, mask


def make_unaugmented_segmentation_dataset(frame):
    dataset = tf.data.Dataset.from_tensor_slices((
        frame.image_path.astype(str).to_numpy(),
        frame.mask_path.astype(str).to_numpy(),
    ))
    dataset = dataset.map(decode_segmentation_pair, num_parallel_calls=AUTOTUNE, deterministic=True)
    return dataset.batch(SEG_BATCH_SIZE, drop_remainder=False).prefetch(AUTOTUNE)


## 10. Paired Augmentation


In [ ]:
# ============================================================
# 10. PAIRED AUGMENTATION
# ============================================================

def _projective_transform(
    image,
    transform,
    interpolation,
    fill_mode,
    fill_value=0.0
):
    transformed = tf.raw_ops.ImageProjectiveTransformV3(
        images=image[None, ...],
        transforms=transform[None, ...],
        output_shape=tf.cast(tf.shape(image)[:2], tf.int32),
        interpolation=interpolation,
        fill_mode=fill_mode,
        fill_value=tf.cast(fill_value, tf.float32),
    )
    return transformed[0]


def stateless_paired_augmentation(index, pair):
    image, mask = pair

    # Reproducible random seed for this image-mask pair
    base_seed = tf.stack([
        tf.cast(SEED, tf.int32),
        tf.cast(index % 2147483647, tf.int32)
    ])

    seeds = tf.random.experimental.stateless_split(base_seed, 9)

    # --------------------------------------------------------
    # 1. Horizontal and vertical flips
    # Image and mask receive EXACTLY the same transformation
    # --------------------------------------------------------

    horizontal = tf.random.stateless_uniform([], seeds[0]) > 0.5
    vertical = tf.random.stateless_uniform([], seeds[1]) > 0.5

    image = tf.cond(
        horizontal,
        lambda: tf.image.flip_left_right(image),
        lambda: image
    )

    mask = tf.cond(
        horizontal,
        lambda: tf.image.flip_left_right(mask),
        lambda: mask
    )

    image = tf.cond(
        vertical,
        lambda: tf.image.flip_up_down(image),
        lambda: image
    )

    mask = tf.cond(
        vertical,
        lambda: tf.image.flip_up_down(mask),
        lambda: mask
    )

    # --------------------------------------------------------
    # 2. Small rotation, zoom and translation
    # --------------------------------------------------------

    angle = tf.random.stateless_uniform(
        [],
        seeds[2],
        minval=-10.0,
        maxval=10.0
    ) * (math.pi / 180.0)

    zoom = tf.random.stateless_uniform(
        [],
        seeds[3],
        minval=0.92,
        maxval=1.08
    )

    tx = tf.random.stateless_uniform(
        [],
        seeds[4],
        minval=-0.05,
        maxval=0.05
    ) * float(SEG_IMAGE_SIZE[1])

    ty = tf.random.stateless_uniform(
        [],
        seeds[5],
        minval=-0.05,
        maxval=0.05
    ) * float(SEG_IMAGE_SIZE[0])

    height, width = map(float, SEG_IMAGE_SIZE)

    cx = (width - 1.0) / 2.0
    cy = (height - 1.0) / 2.0

    cos_a = tf.cos(angle) / zoom
    sin_a = tf.sin(angle) / zoom

    a0 = cos_a
    a1 = sin_a

    b0 = -sin_a
    b1 = cos_a

    a2 = cx - a0 * (cx + tx) - a1 * (cy + ty)
    b2 = cy - b0 * (cx + tx) - b1 * (cy + ty)

    transform = tf.stack([
        a0,
        a1,
        a2,
        b0,
        b1,
        b2,
        0.0,
        0.0
    ])

    # Bilinear interpolation for RGB image
    image = _projective_transform(
        image,
        transform,
        "BILINEAR",
        "REFLECT"
    )

    # Nearest-neighbour interpolation ONLY for mask
    mask = _projective_transform(
        mask,
        transform,
        "NEAREST",
        "CONSTANT",
        0.0
    )

    # --------------------------------------------------------
    # 3. Mild image-only brightness augmentation
    # Deterministic and GPU-compatible
    # --------------------------------------------------------

    brightness_delta = tf.random.stateless_uniform(
        [],
        seed=seeds[6],
        minval=-0.06,
        maxval=0.06,
        dtype=tf.float32
    )

    image = image + brightness_delta

    # --------------------------------------------------------
    # 4. Mild image-only contrast augmentation
    #
    # We calculate contrast manually rather than using
    # tf.image.stateless_random_contrast because AdjustContrastV2
    # caused a deterministic-GPU error in Colab.
    # --------------------------------------------------------

    contrast_factor = tf.random.stateless_uniform(
        [],
        seed=seeds[7],
        minval=0.90,
        maxval=1.10,
        dtype=tf.float32
    )

    image_mean = tf.reduce_mean(
        image,
        axis=[0, 1],
        keepdims=True
    )

    image = (
        (image - image_mean) * contrast_factor
        + image_mean
    )

    # Keep RGB values between 0 and 1
    image = tf.clip_by_value(
        image,
        0.0,
        1.0
    )

    # Ensure mask remains binary
    mask = tf.cast(
        mask >= 0.5,
        tf.float32
    )

    return image, mask


# ============================================================
# CREATE SEGMENTATION DATASETS
# ============================================================

def make_segmentation_dataset(frame, training=False):

    dataset = tf.data.Dataset.from_tensor_slices((
        frame.image_path.astype(str).to_numpy(),
        frame.mask_path.astype(str).to_numpy(),
    ))

    # Shuffle TRAINING data only
    if training:
        dataset = dataset.shuffle(
            len(frame),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    # Decode image-mask pairs
    dataset = dataset.map(
        decode_segmentation_pair,
        num_parallel_calls=AUTOTUNE,
        deterministic=True
    )

    # Apply paired augmentation TRAINING only
    if training:
        dataset = dataset.enumerate()

        dataset = dataset.map(
            stateless_paired_augmentation,
            num_parallel_calls=AUTOTUNE,
            deterministic=True
        )

    # Batch and prefetch
    dataset = dataset.batch(
        SEG_BATCH_SIZE,
        drop_remainder=False
    )

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


# ------------------------------------------------------------
# Create train / validation / test tf.data datasets
# ------------------------------------------------------------

train_seg_ds = make_segmentation_dataset(
    train_seg_df,
    training=True
)

val_seg_ds = make_segmentation_dataset(
    val_seg_df,
    training=False
)

test_seg_ds = make_segmentation_dataset(
    test_seg_df,
    training=False
)


# ============================================================
# AUGMENTATION SANITY CHECK
# ============================================================

example_row = train_seg_df.iloc[0]

original_image, original_mask = decode_segmentation_pair(
    example_row.image_path,
    example_row.mask_path
)

augmented_image, augmented_mask = stateless_paired_augmentation(
    tf.constant(7, tf.int64),
    (original_image, original_mask)
)


fig, axes = plt.subplots(
    1,
    4,
    figsize=(15, 4)
)

for axis, data, title, cmap in [

    (
        axes[0],
        original_image,
        "Original image",
        None
    ),

    (
        axes[1],
        original_mask[..., 0],
        "Original mask",
        "gray"
    ),

    (
        axes[2],
        augmented_image,
        "Augmented image",
        None
    ),

    (
        axes[3],
        augmented_mask[..., 0],
        "Identically augmented mask",
        "gray"
    ),

]:

    axis.imshow(
        data.numpy(),
        cmap=cmap
    )

    axis.set_title(title)
    axis.axis("off")


plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "paired_augmentation_check.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# DATASET SHAPE CHECK
# ============================================================

sample_images, sample_masks = next(iter(train_seg_ds))

print("Training image batch shape:", sample_images.shape)
print("Training mask batch shape:", sample_masks.shape)

assert sample_images.shape[1:] == (256, 256, 3)
assert sample_masks.shape[1:] == (256, 256, 1)

print("Paired augmentation and tf.data segmentation pipeline passed.")

## 11. Segmentation Metrics and Loss


In [ ]:
EPSILON = 1e-7


@keras.utils.register_keras_serializable(package="MelanomaAI")
def dice_coefficient(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    axes = tuple(range(1, len(y_true.shape)))
    intersection = tf.reduce_sum(y_true * y_pred, axis=axes)
    denominator = tf.reduce_sum(y_true, axis=axes) + tf.reduce_sum(y_pred, axis=axes)
    return tf.reduce_mean((2.0 * intersection + EPSILON) / (denominator + EPSILON))


@keras.utils.register_keras_serializable(package="MelanomaAI")
def iou_coefficient(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    axes = tuple(range(1, len(y_true.shape)))
    intersection = tf.reduce_sum(y_true * y_pred, axis=axes)
    union = tf.reduce_sum(y_true + y_pred - y_true * y_pred, axis=axes)
    return tf.reduce_mean((intersection + EPSILON) / (union + EPSILON))


@keras.utils.register_keras_serializable(package="MelanomaAI")
def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(y_true, y_pred)


@keras.utils.register_keras_serializable(package="MelanomaAI")
def combined_segmentation_loss(y_true, y_pred):
    bce = tf.reduce_mean(keras.losses.binary_crossentropy(y_true, y_pred))
    return bce + dice_loss(y_true, y_pred)


metric_test_cases = [
    (tf.zeros((2, 16, 16, 1)), tf.zeros((2, 16, 16, 1))),
    (tf.ones((2, 16, 16, 1)), tf.ones((2, 16, 16, 1))),
    (tf.ones((2, 16, 16, 1)), tf.zeros((2, 16, 16, 1))),
]
for true_mask, predicted_mask in metric_test_cases:
    values = [
        dice_coefficient(true_mask, predicted_mask),
        iou_coefficient(true_mask, predicted_mask),
        dice_loss(true_mask, predicted_mask),
        combined_segmentation_loss(true_mask, predicted_mask),
    ]
    assert all(np.isfinite(float(value.numpy())) for value in values)
print("Dice, IoU, Dice loss and combined-loss synthetic tests passed.")


## 12. Residual Block


A residual block has a two-convolution learning path and a shortcut path. If the input channel count differs from the requested filters, a 1×1 convolution plus batch normalisation projects the shortcut to the correct shape before addition. No broadcasting is used.


In [ ]:
def residual_block(x, filters, name):
    shortcut = x
    residual = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv1")(x)
    residual = layers.BatchNormalization(name=f"{name}_bn1")(residual)
    residual = layers.Activation("relu", name=f"{name}_relu1")(residual)
    residual = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv2")(residual)
    residual = layers.BatchNormalization(name=f"{name}_bn2")(residual)

    input_channels = shortcut.shape[-1]
    if input_channels != filters:
        shortcut = layers.Conv2D(filters, 1, padding="same", use_bias=False, name=f"{name}_shortcut_conv")(shortcut)
        shortcut = layers.BatchNormalization(name=f"{name}_shortcut_bn")(shortcut)

    output = layers.Add(name=f"{name}_add")([residual, shortcut])
    return layers.Activation("relu", name=f"{name}_out_relu")(output)


## 13. ResUNet Architecture


Standard U-Net uses ordinary convolution blocks. **ResUNet** replaces them with residual blocks, allowing information and gradients to travel through shortcut paths. This may improve gradient flow, reduce degradation in deeper networks, preserve useful lower-level features and support lesion-boundary learning. These are motivations, not a claim that ResUNet is already superior. Its validation and held-out test results must be compared experimentally with the team's separate Standard U-Net, Attention U-Net and U-Net++ branches under the same conditions.


In [ ]:
def build_resunet(input_shape=(*SEG_IMAGE_SIZE, 3)):
    inputs = keras.Input(shape=input_shape, name="dermoscopic_image")

    encoder_1 = residual_block(inputs, 32, "encoder_1")
    pooled_1 = layers.MaxPooling2D(2, name="pool_1")(encoder_1)
    encoder_2 = residual_block(pooled_1, 64, "encoder_2")
    pooled_2 = layers.MaxPooling2D(2, name="pool_2")(encoder_2)
    encoder_3 = residual_block(pooled_2, 128, "encoder_3")
    pooled_3 = layers.MaxPooling2D(2, name="pool_3")(encoder_3)
    encoder_4 = residual_block(pooled_3, 256, "encoder_4")
    pooled_4 = layers.MaxPooling2D(2, name="pool_4")(encoder_4)

    bottleneck = residual_block(pooled_4, 512, "bottleneck")

    def decoder_block(x, skip, filters, name):
        x = layers.Conv2DTranspose(filters, 2, strides=2, padding="same", name=f"{name}_upsample")(x)
        x = layers.Concatenate(name=f"{name}_skip_concat")([x, skip])
        return residual_block(x, filters, name)

    decoder_4 = decoder_block(bottleneck, encoder_4, 256, "decoder_4")
    decoder_3 = decoder_block(decoder_4, encoder_3, 128, "decoder_3")
    decoder_2 = decoder_block(decoder_3, encoder_2, 64, "decoder_2")
    decoder_1 = decoder_block(decoder_2, encoder_1, 32, "decoder_1")

    outputs = layers.Conv2D(
        filters=1,
        kernel_size=1,
        activation="sigmoid",
        dtype="float32",
        name="lesion_probability",
    )(decoder_1)
    return keras.Model(inputs, outputs, name="ResUNet")


## 14. ResUNet Self-Test


In [ ]:
keras.backend.clear_session()
resunet = build_resunet()
resunet.summary()
print("ResUNet parameter count:", f"{resunet.count_params():,}")

dummy = tf.zeros((1, 256, 256, 3), dtype=tf.float32)
dummy_output = resunet(dummy, training=False)
assert tuple(dummy_output.shape) == (1, 256, 256, 1)

projection_input = keras.Input(shape=(32, 32, 32))
projection_model = keras.Model(projection_input, residual_block(projection_input, 64, "projection_test"))
projection_output = projection_model(tf.zeros((1, 32, 32, 32)), training=False)
assert tuple(projection_output.shape) == (1, 32, 32, 64)
assert projection_model.get_layer("projection_test_shortcut_conv").filters == 64
print("ResUNet forward pass and 32→64 shortcut-projection tests passed.")
del projection_model, projection_input, projection_output, dummy, dummy_output
gc.collect()


## 15. ResUNet Training


In [ ]:
CUSTOM_OBJECTS = {
    "dice_coefficient": dice_coefficient,
    "iou_coefficient": iou_coefficient,
    "dice_loss": dice_loss,
    "combined_segmentation_loss": combined_segmentation_loss,
}


class LearningRateHistory(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs is not None:
            logs["learning_rate"] = float(keras.backend.get_value(self.model.optimizer.learning_rate))


def compile_resunet(model):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=SEG_LEARNING_RATE),
        loss=combined_segmentation_loss,
        metrics=[dice_coefficient, iou_coefficient],
    )
    return model


RESUNET_BEST_PATH = OUTPUT_DIR / "resunet_best.keras"
resunet_callbacks = [
    keras.callbacks.ModelCheckpoint(
        RESUNET_BEST_PATH,
        monitor="val_dice_coefficient",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_dice_coefficient",
        mode="max",
        patience=10,
        restore_best_weights=False,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_dice_coefficient",
        mode="max",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1,
    ),
    LearningRateHistory(),
]

keras.utils.set_random_seed(SEED)
resunet = compile_resunet(resunet)
segmentation_training_started = time.perf_counter()
resunet_history_object = resunet.fit(
    train_seg_ds,
    validation_data=val_seg_ds,
    epochs=SEG_EPOCHS,
    callbacks=resunet_callbacks,
    verbose=1,
)
SEGMENTATION_TRAINING_SECONDS = time.perf_counter() - segmentation_training_started

resunet_history = pd.DataFrame(resunet_history_object.history)
resunet_history.index = np.arange(1, len(resunet_history) + 1)
resunet_history.index.name = "epoch"
resunet_history.to_csv(OUTPUT_DIR / "resunet_training_history.csv")
print(f"ResUNet training time: {SEGMENTATION_TRAINING_SECONDS / 60:.2f} minutes")


## 16. Training Curves


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for axis, metric, title in [
    (axes[0], "loss", "Training vs Validation Loss"),
    (axes[1], "dice_coefficient", "Training vs Validation Dice"),
    (axes[2], "iou_coefficient", "Training vs Validation IoU"),
]:
    axis.plot(resunet_history.index, resunet_history[metric], label="Training")
    axis.plot(resunet_history.index, resunet_history[f"val_{metric}"], label="Validation")
    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.grid(alpha=0.25)
    axis.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "resunet_training_curves.png", dpi=180, bbox_inches="tight")
plt.show()


## 17. ResUNet Validation/Test Results


In [ ]:
# ============================================================
# 17. RESUNET VALIDATION / TEST RESULTS
# VALIDATION-SELECTED SEGMENTATION THRESHOLD
# ============================================================

# Reload the BEST checkpoint selected during training
resunet_best = keras.models.load_model(
    RESUNET_BEST_PATH,
    custom_objects=CUSTOM_OBJECTS,
    compile=False
)

# ------------------------------------------------------------
# 1. Collect validation probabilities
# ------------------------------------------------------------

val_probability_batches = []
val_truth_batches = []

for images, true_masks in val_seg_ds:
    probabilities = resunet_best.predict_on_batch(images).astype(np.float32)

    val_probability_batches.append(probabilities)
    val_truth_batches.append(true_masks.numpy().astype(np.float32))

val_probabilities = np.concatenate(val_probability_batches, axis=0)
val_truth = np.concatenate(val_truth_batches, axis=0)

print("Validation prediction probability range:")
print("Minimum:", float(val_probabilities.min()))
print("Maximum:", float(val_probabilities.max()))
print("Mean:", float(val_probabilities.mean()))


# ------------------------------------------------------------
# 2. Find best threshold using VALIDATION DATA ONLY
# ------------------------------------------------------------

def mean_binary_dice(probabilities, truth, threshold):

    predicted = probabilities >= threshold
    actual = truth >= 0.5

    axes = (1, 2, 3)

    intersection = np.sum(
        predicted & actual,
        axis=axes
    )

    predicted_sum = np.sum(
        predicted,
        axis=axes
    )

    truth_sum = np.sum(
        actual,
        axis=axes
    )

    dice = (
        2 * intersection + EPSILON
    ) / (
        predicted_sum + truth_sum + EPSILON
    )

    return float(np.mean(dice))


threshold_candidates = np.arange(
    0.05,
    0.51,
    0.01
)

threshold_results = []

for threshold in threshold_candidates:

    validation_dice = mean_binary_dice(
        val_probabilities,
        val_truth,
        threshold
    )

    threshold_results.append({
        "threshold": float(threshold),
        "validation_dice": validation_dice
    })


threshold_table = pd.DataFrame(
    threshold_results
)

best_threshold_row = threshold_table.loc[
    threshold_table["validation_dice"].idxmax()
]

SELECTED_SEG_THRESHOLD = float(
    best_threshold_row["threshold"]
)

BEST_THRESHOLD_VALIDATION_DICE = float(
    best_threshold_row["validation_dice"]
)

print()
print(
    "Selected segmentation threshold:",
    round(SELECTED_SEG_THRESHOLD, 3)
)

print(
    "Validation Dice at selected threshold:",
    round(BEST_THRESHOLD_VALIDATION_DICE, 4)
)

display(
    threshold_table.sort_values(
        "validation_dice",
        ascending=False
    ).head(10)
)


# Save the threshold chosen from validation only
seg_threshold_payload = {
    "default_threshold": float(SEG_THRESHOLD),
    "selected_threshold": float(SELECTED_SEG_THRESHOLD),
    "selection_data": "validation_only",
    "best_validation_dice": float(BEST_THRESHOLD_VALIDATION_DICE),
}

(
    OUTPUT_DIR /
    "segmentation_threshold.json"
).write_text(
    json.dumps(
        seg_threshold_payload,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 3. Evaluation function
# ------------------------------------------------------------

def evaluate_segmenter_streaming(
    model,
    dataset,
    threshold
):

    dice_values = []
    iou_values = []

    tp = 0
    tn = 0
    fp = 0
    fn = 0

    for images, true_masks in tqdm(
        dataset,
        desc="Evaluating ResUNet",
        leave=False
    ):

        probabilities = model.predict_on_batch(
            images
        ).astype(np.float32)

        predicted = (
            probabilities >= threshold
        )

        truth = (
            true_masks.numpy() >= 0.5
        )

        axes = (1, 2, 3)

        intersection = np.sum(
            predicted & truth,
            axis=axes
        )

        predicted_sum = np.sum(
            predicted,
            axis=axes
        )

        truth_sum = np.sum(
            truth,
            axis=axes
        )

        union = np.sum(
            predicted | truth,
            axis=axes
        )

        dice_values.extend(
            (
                (
                    2 * intersection +
                    EPSILON
                ) /
                (
                    predicted_sum +
                    truth_sum +
                    EPSILON
                )
            ).tolist()
        )

        iou_values.extend(
            (
                (
                    intersection +
                    EPSILON
                ) /
                (
                    union +
                    EPSILON
                )
            ).tolist()
        )

        tp += int(
            np.sum(predicted & truth)
        )

        tn += int(
            np.sum(
                (~predicted) &
                (~truth)
            )
        )

        fp += int(
            np.sum(
                predicted &
                (~truth)
            )
        )

        fn += int(
            np.sum(
                (~predicted) &
                truth
            )
        )

    metrics = {

        "mean_dice":
            float(np.mean(dice_values)),

        "mean_iou":
            float(np.mean(iou_values)),

        "pixel_sensitivity":
            float(
                (tp + EPSILON) /
                (tp + fn + EPSILON)
            ),

        "pixel_specificity":
            float(
                (tn + EPSILON) /
                (tn + fp + EPSILON)
            ),

        "tp_pixels": tp,
        "tn_pixels": tn,
        "fp_pixels": fp,
        "fn_pixels": fn,
    }

    return (
        metrics,
        np.asarray(dice_values),
        np.asarray(iou_values)
    )


# ------------------------------------------------------------
# 4. Evaluate validation and FINAL TEST using locked threshold
# ------------------------------------------------------------

validation_segmentation_metrics, \
validation_dice_values, \
validation_iou_values = evaluate_segmenter_streaming(

    resunet_best,
    val_seg_ds,
    SELECTED_SEG_THRESHOLD
)


test_segmentation_metrics, \
test_dice_values, \
test_iou_values = evaluate_segmenter_streaming(

    resunet_best,
    test_seg_ds,
    SELECTED_SEG_THRESHOLD
)


# ------------------------------------------------------------
# 5. Save per-case test results
# ------------------------------------------------------------

segmentation_case_scores = test_seg_df[
    [
        "image_id",
        "image_path",
        "mask_path"
    ]
].copy()

segmentation_case_scores[
    "dice"
] = test_dice_values

segmentation_case_scores[
    "iou"
] = test_iou_values

segmentation_case_scores.to_csv(
    OUTPUT_DIR /
    "resunet_test_case_scores.csv",
    index=False
)


# ------------------------------------------------------------
# 6. Final result table
# ------------------------------------------------------------

resunet_results = pd.DataFrame([{

    "Model":
        "ResUNet",

    "Segmentation Threshold":
        SELECTED_SEG_THRESHOLD,

    "Validation Dice":
        validation_segmentation_metrics[
            "mean_dice"
        ],

    "Validation IoU":
        validation_segmentation_metrics[
            "mean_iou"
        ],

    "Test Dice":
        test_segmentation_metrics[
            "mean_dice"
        ],

    "Test IoU":
        test_segmentation_metrics[
            "mean_iou"
        ],

    "Pixel Sensitivity":
        test_segmentation_metrics[
            "pixel_sensitivity"
        ],

    "Pixel Specificity":
        test_segmentation_metrics[
            "pixel_specificity"
        ],

    "TP Pixels":
        test_segmentation_metrics[
            "tp_pixels"
        ],

    "TN Pixels":
        test_segmentation_metrics[
            "tn_pixels"
        ],

    "FP Pixels":
        test_segmentation_metrics[
            "fp_pixels"
        ],

    "FN Pixels":
        test_segmentation_metrics[
            "fn_pixels"
        ],

    "Parameter Count":
        resunet_best.count_params(),

    "Training Time (seconds)":
        SEGMENTATION_TRAINING_SECONDS,

}])


resunet_results.to_csv(
    OUTPUT_DIR /
    "resunet_results.csv",
    index=False
)

display(
    resunet_results.T
)


# ------------------------------------------------------------
# 7. Lock selected model
# ------------------------------------------------------------

SELECTED_SEGMENTER_PATH = (
    OUTPUT_DIR /
    "selected_segmenter.keras"
)

shutil.copy2(
    RESUNET_BEST_PATH,
    SELECTED_SEGMENTER_PATH
)

selected_segmenter = keras.models.load_model(
    SELECTED_SEGMENTER_PATH,
    custom_objects=CUSTOM_OBJECTS,
    compile=False
)

print()
print(
    "Threshold was selected using VALIDATION data only."
)

print(
    "Final locked segmentation threshold:",
    SELECTED_SEG_THRESHOLD
)

print(
    "The TEST split was evaluated only after threshold selection."
)

## 18. Qualitative Segmentation Results


In [ ]:
# ============================================================
# GENERATE ONE RESUNET PROBABILITY MASK
# ============================================================

def predict_resized_segmentation(model, image_pil):
    """
    Prepare one image for ResUNet and return its probability mask.
    """

    # Resize the image to the input size expected by ResUNet
    resized_image = image_pil.resize(
        (
            SEG_IMAGE_SIZE[1],
            SEG_IMAGE_SIZE[0]
        ),
        Image.Resampling.BILINEAR
    )

    # Convert the image to an array and scale pixels to 0–1
    model_input = np.asarray(
        resized_image,
        dtype=np.float32
    ) / 255.0

    # Add the batch dimension
    model_input = model_input[None, ...]

    # Generate the predicted probability mask
    probability_mask = model.predict(
        model_input,
        verbose=0
    )[0, :, :, 0]

    return probability_mask


# ============================================================
# DISPLAY GROUND TRUTH AND RESUNET PREDICTION
# ============================================================

def display_segmentation_cases(
    rows,
    figure_name,
    heading=None
):
    """
    Display:
    1. Original image
    2. Expert ground-truth mask
    3. ResUNet prediction
    4. ResUNet prediction overlay
    """

    rows = rows.reset_index(drop=True)

    if heading is not None:
        print(heading)

    fig, axes = plt.subplots(
        nrows=len(rows),
        ncols=4,
        figsize=(15, 3.6 * len(rows))
    )

    axes = np.atleast_2d(axes)


    for row_index, row in rows.iterrows():

        # ----------------------------------------------------
        # LOAD THE ORIGINAL IMAGE
        # ----------------------------------------------------

        original_image = load_rgb_pil(
            row["image_path"]
        ).convert("RGB")

        # Resize the original image for display
        resized_image = original_image.resize(
            (
                SEG_IMAGE_SIZE[1],
                SEG_IMAGE_SIZE[0]
            ),
            Image.Resampling.BILINEAR
        )

        # ----------------------------------------------------
        # LOAD THE EXPERT GROUND-TRUTH MASK
        # ----------------------------------------------------

        ground_truth_image = load_mask_pil(
            row["mask_path"]
        ).convert("L")

        ground_truth_image = ground_truth_image.resize(
            (
                SEG_IMAGE_SIZE[1],
                SEG_IMAGE_SIZE[0]
            ),
            Image.Resampling.NEAREST
        )

        # Convert to a binary mask
        ground_truth = (
            np.asarray(ground_truth_image) > 127
        )

        # ----------------------------------------------------
        # GENERATE THE RESUNET PREDICTION
        # ----------------------------------------------------

        prediction_probability = predict_resized_segmentation(
            selected_segmenter,
            original_image
        )

        # Convert probability values into a binary mask
        prediction = (
            prediction_probability
            >= SELECTED_SEG_THRESHOLD
        )

        # ----------------------------------------------------
        # CALCULATE DICE
        # ----------------------------------------------------

        intersection = np.logical_and(
            ground_truth,
            prediction
        ).sum()

        dice = (
            2.0 * intersection + EPSILON
        ) / (
            ground_truth.sum()
            + prediction.sum()
            + EPSILON
        )

        # ----------------------------------------------------
        # CALCULATE IOU
        # ----------------------------------------------------

        union = np.logical_or(
            ground_truth,
            prediction
        ).sum()

        iou = (
            intersection + EPSILON
        ) / (
            union + EPSILON
        )

        # ----------------------------------------------------
        # COLUMN 1: ORIGINAL IMAGE
        # ----------------------------------------------------

        axes[row_index, 0].imshow(
            resized_image
        )

        axes[row_index, 0].set_title(
            f"Original image\n{row['image_id']}"
        )

        # ----------------------------------------------------
        # COLUMN 2: EXPERT GROUND-TRUTH MASK
        # ----------------------------------------------------

        axes[row_index, 1].imshow(
            ground_truth,
            cmap="gray",
            vmin=0,
            vmax=1
        )

        axes[row_index, 1].set_title(
            "Expert ground-truth mask"
        )

        # ----------------------------------------------------
        # COLUMN 3: RESUNET PREDICTED MASK
        # ----------------------------------------------------

        axes[row_index, 2].imshow(
            prediction,
            cmap="gray",
            vmin=0,
            vmax=1
        )

        axes[row_index, 2].set_title(
            "ResUNet prediction\n"
            f"Dice = {dice:.3f} | IoU = {iou:.3f}"
        )

        # ----------------------------------------------------
        # COLUMN 4: RESUNET PREDICTION OVERLAY
        # ----------------------------------------------------

        show_mask_overlay(
            axis=axes[row_index, 3],
            image=resized_image,
            mask=prediction,
            red=1.0,
            green=0.0,
            blue=0.0,
            alpha=0.35
        )

        axes[row_index, 3].set_title(
            "Prediction overlay"
        )

        # Remove axis lines and numbers
        for axis in axes[row_index]:
            axis.axis("off")


    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR / figure_name,
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()


# ============================================================
# SELECT AND DISPLAY FIVE TEST IMAGES
# ============================================================

representative_cases = segmentation_case_scores.sample(
    n=min(5, len(segmentation_case_scores)),
    random_state=SEED
)

display_segmentation_cases(
    rows=representative_cases,
    figure_name="resunet_qualitative_test_examples.png",
    heading="Representative final-test segmentations"
)

## 19. Best/Median/Worst Results


In [ ]:
sorted_scores = segmentation_case_scores.sort_values("dice").reset_index(drop=True)
selected_positions = [len(sorted_scores) - 1, len(sorted_scores) // 2, 0]
best_median_worst = sorted_scores.iloc[selected_positions].copy()
best_median_worst["case_type"] = ["Best Dice", "Median Dice", "Worst Dice"]
print(best_median_worst[["case_type", "image_id", "dice", "iou"]].to_string(index=False))
display_segmentation_cases(
    best_median_worst,
    "resunet_best_median_worst.png",
    "Best, median and worst final-test segmentations",
)


## 20. Segmentation Error Analysis


The lowest-Dice images below should be inspected visually. Possible *observable* contributors include hair crossing the lesion, ruler markings, gel bubbles or specular reflections, low lesion-to-skin contrast, poorly defined borders, lighting variation, or multiple pigmented regions. The notebook does not automatically assign a cause because these factors require human inspection and may coexist.


In [ ]:
poor_cases = sorted_scores.head(min(5, len(sorted_scores)))
display_segmentation_cases(poor_cases, "resunet_lowest_dice_error_analysis.png")
display(poor_cases[["image_id", "dice", "iou"]])
print("Inspect only visible factors; do not infer an unobserved cause from Dice alone.")


## 21. ResUNet Team Comparison Export


In [ ]:
team_comparison_row = resunet_results[[
    "Model",
    "Validation Dice",
    "Validation IoU",
    "Test Dice",
    "Test IoU",
    "Pixel Sensitivity",
    "Pixel Specificity",
    "Parameter Count",
    "Training Time (seconds)",
]].copy()
team_comparison_row.to_csv(OUTPUT_DIR / "ResUNet_team_comparison_row.csv", index=False)
display(team_comparison_row)
print("This single row can be combined with Standard U-Net, Attention U-Net and U-Net++ results.")


## 22. SIIM-ISIC 2020 Dataset


Only the official labelled SIIM-ISIC 2020 training data are used. Public test labels are unavailable, so this notebook creates an internal patient-disjoint train/validation/final-test split. The code prints the actual CSV columns before selecting fields and removes the second image in every pair listed by the official duplicate CSV.


In [ ]:
download_extract_and_cleanup("cls_images", CLS_WORK_DIR / "images")

dataset_files = {}
for dataset_key in ["cls_ground_truth", "cls_metadata_v2", "cls_duplicates"]:
    dataset_files[dataset_key] = download_with_resume(DATASET_SPECS[dataset_key], destination_dir=ARCHIVE_DIR)

ground_truth = pd.read_csv(dataset_files["cls_ground_truth"])
metadata_v2 = pd.read_csv(dataset_files["cls_metadata_v2"])
duplicate_pairs = pd.read_csv(dataset_files["cls_duplicates"])

print("Ground-truth columns:", ground_truth.columns.tolist())
print("Metadata-v2 columns:", metadata_v2.columns.tolist())
print("Duplicate-file columns:", duplicate_pairs.columns.tolist())

required_ground_truth = {"image_name", "target"}
required_metadata = {"image_name", "patient_id"}
if not required_ground_truth.issubset(ground_truth.columns):
    raise RuntimeError(f"Ground-truth CSV is missing {sorted(required_ground_truth - set(ground_truth.columns))}.")
if not required_metadata.issubset(metadata_v2.columns):
    raise RuntimeError(f"Metadata-v2 CSV is missing {sorted(required_metadata - set(metadata_v2.columns))}.")
if ground_truth.image_name.duplicated().any() or metadata_v2.image_name.duplicated().any():
    raise RuntimeError("Unexpected duplicate image_name rows in an official metadata table.")

available_metadata_columns = [
    column for column in [
        "image_name", "patient_id", "lesion_id", "sex", "age_approx",
        "anatom_site_general_challenge", "diagnosis", "benign_malignant",
    ] if column in metadata_v2.columns
]
classification_manifest = metadata_v2[available_metadata_columns].merge(
    ground_truth[["image_name", "target"]], on="image_name", how="inner", validate="one_to_one"
)
classification_manifest["target"] = pd.to_numeric(classification_manifest.target, errors="coerce")
if classification_manifest[["image_name", "patient_id", "target"]].isna().any().any():
    raise RuntimeError("Required classification fields contain missing/non-numeric values.")
classification_manifest["target"] = classification_manifest.target.astype(int)
if not set(classification_manifest.target.unique()).issubset({0, 1}):
    raise RuntimeError("Classification target is not binary 0/1.")
if len(classification_manifest) != EXPECTED_CLASSIFICATION_IMAGES:
    raise RuntimeError(f"Expected {EXPECTED_CLASSIFICATION_IMAGES} labelled images; found {len(classification_manifest)}.")

classification_image_paths = [
    path for path in (CLS_WORK_DIR / "images").rglob("*")
    if path.suffix.lower() in {".jpg", ".jpeg"}
]
classification_path_table = pd.DataFrame({
    "image_name": [path.stem for path in classification_image_paths],
    "original_path": [str(path) for path in classification_image_paths],
})
if classification_path_table.image_name.duplicated().any():
    raise RuntimeError("Duplicate image filenames were found after SIIM-ISIC extraction.")
classification_manifest = classification_manifest.merge(
    classification_path_table, on="image_name", how="left", validate="one_to_one"
)
if classification_manifest.original_path.isna().any():
    missing = classification_manifest.loc[classification_manifest.original_path.isna(), "image_name"].head(10).tolist()
    raise RuntimeError(f"Labelled images were not found on disk. Examples: {missing}")

if not {"image_name_1", "image_name_2"}.issubset(duplicate_pairs.columns):
    raise RuntimeError("The official duplicate CSV does not contain image_name_1 and image_name_2.")
duplicate_ids_to_remove = set(duplicate_pairs.image_name_2.astype(str))
unknown_duplicates = duplicate_ids_to_remove - set(classification_manifest.image_name)
if unknown_duplicates:
    raise RuntimeError(f"Duplicate IDs absent from the manifest: {sorted(unknown_duplicates)[:10]}")

classification_clean = classification_manifest.loc[
    ~classification_manifest.image_name.isin(duplicate_ids_to_remove)
].copy().reset_index(drop=True)
classification_clean["image_id"] = classification_clean.image_name
classification_clean["patient_id"] = classification_clean.patient_id.astype(str)
classification_manifest.to_csv(OUTPUT_DIR / "classification_manifest_before_duplicate_removal.csv", index=False)
classification_clean.to_csv(OUTPUT_DIR / "classification_manifest_duplicate_clean.csv", index=False)
duplicate_pairs.to_csv(OUTPUT_DIR / "official_duplicate_pairs_used.csv", index=False)

print("Official labelled images:", len(classification_manifest))
print("Duplicate copies removed:", len(duplicate_ids_to_remove))
print("Images after duplicate removal:", len(classification_clean))
print("Patients:", classification_clean.patient_id.nunique())
print("Class counts:", classification_clean.target.value_counts().sort_index().to_dict())


## 23. Patient-Level Classification Split


In [ ]:
def choose_best_group_holdout(frame, desired_fraction, n_splits, seed):
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    overall_prevalence = frame.target.mean()
    candidates = []
    for fold_number, (keep_indices, holdout_indices) in enumerate(
        splitter.split(frame, frame.target, groups=frame.patient_id)
    ):
        holdout = frame.iloc[holdout_indices]
        fraction_error = abs(len(holdout) / len(frame) - desired_fraction)
        prevalence_error = abs(holdout.target.mean() - overall_prevalence)
        candidates.append((fraction_error + prevalence_error, fold_number, keep_indices, holdout_indices))
    _, selected_fold, keep_indices, holdout_indices = min(candidates, key=lambda item: (item[0], item[1]))
    print(f"Selected fold {selected_fold} of {n_splits} for desired fraction {desired_fraction:.3f}.")
    return keep_indices, holdout_indices


# Seven folds gives approximately 14.3%; the scoring also favours prevalence similarity.
trainval_indices, test_indices = choose_best_group_holdout(
    classification_clean, desired_fraction=0.15, n_splits=7, seed=SEED
)
classification_trainval = classification_clean.iloc[trainval_indices].reset_index(drop=True)
classification_test = classification_clean.iloc[test_indices].reset_index(drop=True)

# Validation is approximately 15% of the original data (15/85 of the remainder).
train_indices, validation_indices = choose_best_group_holdout(
    classification_trainval, desired_fraction=0.15 / 0.85, n_splits=6, seed=SEED + 1
)
classification_train = classification_trainval.iloc[train_indices].reset_index(drop=True)
classification_validation = classification_trainval.iloc[validation_indices].reset_index(drop=True)

train_patients = set(classification_train.patient_id)
validation_patients = set(classification_validation.patient_id)
test_patients = set(classification_test.patient_id)
assert train_patients.isdisjoint(validation_patients)
assert train_patients.isdisjoint(test_patients)
assert validation_patients.isdisjoint(test_patients)
assert len(classification_train) + len(classification_validation) + len(classification_test) == len(classification_clean)

for split_name, split_frame in [
    ("train", classification_train),
    ("validation", classification_validation),
    ("test", classification_test),
]:
    if set(split_frame.target.unique()) != {0, 1}:
        raise RuntimeError(f"The {split_name} split does not contain both classes.")


def classification_split_summary(split_name, frame):
    benign = int((frame.target == 0).sum())
    malignant = int((frame.target == 1).sum())
    return {
        "split": split_name,
        "images": len(frame),
        "patients": frame.patient_id.nunique(),
        "benign": benign,
        "malignant": malignant,
        "benign_percent": 100.0 * benign / len(frame),
        "malignant_percent": 100.0 * malignant / len(frame),
    }


classification_split_summary_table = pd.DataFrame([
    classification_split_summary("train", classification_train),
    classification_split_summary("validation", classification_validation),
    classification_split_summary("test", classification_test),
])
display(classification_split_summary_table.round(3))

classification_train.to_csv(OUTPUT_DIR / "classification_train_split.csv", index=False)
classification_validation.to_csv(OUTPUT_DIR / "classification_validation_split.csv", index=False)
classification_test.to_csv(OUTPUT_DIR / "classification_test_split.csv", index=False)
classification_split_summary_table.to_csv(OUTPUT_DIR / "classification_split_summary.csv", index=False)
print("ZERO patient overlap confirmed across train, validation and final test.")


## 24. ResUNet Inference on Classification Dataset


In [ ]:
def find_unreadable_images(frame):
    unreadable = []
    for row in tqdm(frame[["image_name", "original_path"]].itertuples(index=False), total=len(frame), desc="Checking SIIM-ISIC files"):
        try:
            with Image.open(row.original_path) as image:
                image.verify()
        except (OSError, ValueError, UnidentifiedImageError) as exc:
            unreadable.append({"image_name": row.image_name, "path": row.original_path, "error": repr(exc)})
    return pd.DataFrame(unreadable, columns=["image_name", "path", "error"])


if VERIFY_IMAGE_CONTENT:
    unreadable_classification_images = find_unreadable_images(classification_clean)
else:
    unreadable_classification_images = pd.DataFrame(columns=["image_name", "path", "error"])
unreadable_classification_images.to_csv(OUTPUT_DIR / "classification_unreadable_images.csv", index=False)
if len(unreadable_classification_images):
    raise RuntimeError("Unreadable official training images were found; inspect classification_unreadable_images.csv.")


def decode_segmentation_inference(path):
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    image = tf.image.resize(image, SEG_IMAGE_SIZE, method="bilinear")
    image = tf.cast(image, tf.float32) / 255.0
    image.set_shape((*SEG_IMAGE_SIZE, 3))
    return path, image


def make_segmentation_inference_dataset(paths):
    dataset = tf.data.Dataset.from_tensor_slices(np.asarray(paths, dtype=str))
    dataset = dataset.map(decode_segmentation_inference, num_parallel_calls=AUTOTUNE, deterministic=True)
    return dataset.batch(SEG_BATCH_SIZE, drop_remainder=False).prefetch(AUTOTUNE)


print("ResUNet batched inference pipeline is ready. Masks/ROIs will be generated once in Section 25.")


## 25. Lesion ROI Generation


The main segmentation comparison used raw masks thresholded at 0.5. ROI extraction also starts from the 0.5 mask. Optional largest-component/hole-filling logic is controlled by `ROI_POSTPROCESS` and affects only ROI generation, never the reported team-comparison segmentation metrics.


In [ ]:
def prepare_mask_for_roi(probability_mask, threshold=SEG_THRESHOLD):
    binary = np.asarray(probability_mask, dtype=np.float32) >= float(threshold)
    if ROI_POSTPROCESS and np.any(binary):
        labels, count = ndimage.label(binary)
        if count > 1:
            sizes = ndimage.sum(binary, labels, index=np.arange(1, count + 1))
            binary = labels == (int(np.argmax(sizes)) + 1)
        binary = ndimage.binary_fill_holes(binary)
    return binary.astype(np.uint8)


def extract_lesion_roi(image, predicted_mask, padding_fraction=ROI_PADDING_FRACTION):
    image_array = np.asarray(image.convert("RGB") if isinstance(image, Image.Image) else image)
    if image_array.ndim != 3 or image_array.shape[-1] != 3:
        raise ValueError("image must have RGB shape H×W×3")
    mask = np.asarray(predicted_mask).squeeze().astype(bool)
    height, width = image_array.shape[:2]
    if mask.shape != (height, width):
        mask = np.asarray(
            Image.fromarray(mask.astype(np.uint8) * 255).resize((width, height), Image.Resampling.NEAREST)
        ) > 127

    mask_fraction = float(mask.mean())
    if not np.any(mask):
        return image_array.copy(), (0, 0, width, height), True, "empty_mask"
    if mask_fraction < MIN_VALID_MASK_FRACTION:
        return image_array.copy(), (0, 0, width, height), True, "mask_too_small"
    if mask_fraction > MAX_VALID_MASK_FRACTION:
        return image_array.copy(), (0, 0, width, height), True, "mask_too_large"

    y_coordinates, x_coordinates = np.where(mask)
    x0, x1 = int(x_coordinates.min()), int(x_coordinates.max()) + 1
    y0, y1 = int(y_coordinates.min()), int(y_coordinates.max()) + 1
    padding_x = max(1, int(round((x1 - x0) * padding_fraction)))
    padding_y = max(1, int(round((y1 - y0) * padding_fraction)))
    x0, x1 = max(0, x0 - padding_x), min(width, x1 + padding_x)
    y0, y1 = max(0, y0 - padding_y), min(height, y1 + padding_y)
    if x1 <= x0 or y1 <= y0:
        return image_array.copy(), (0, 0, width, height), True, "invalid_bounding_box"
    if (x1 - x0) < MIN_ROI_SIDE_PIXELS or (y1 - y0) < MIN_ROI_SIDE_PIXELS:
        return image_array.copy(), (0, 0, width, height), True, "crop_too_small"
    return image_array[y0:y1, x0:x1].copy(), (x0, y0, x1, y1), False, None


# Synthetic ROI tests: normal, empty and full-image masks.
synthetic_image = Image.fromarray(np.full((100, 120, 3), 128, dtype=np.uint8))
normal_mask = np.zeros((100, 120), dtype=np.uint8)
normal_mask[30:70, 40:90] = 1
empty_mask = np.zeros((100, 120), dtype=np.uint8)
full_image_mask = np.ones((100, 120), dtype=np.uint8)
assert extract_lesion_roi(synthetic_image, normal_mask)[2] is False
assert extract_lesion_roi(synthetic_image, empty_mask)[2] is True
assert extract_lesion_roi(synthetic_image, full_image_mask)[2] is True
print("Normal, empty-mask and full-image-mask ROI tests passed.")


def generate_roi_manifest(frame, segmenter, checkpoint_path):
    frame = frame.copy().reset_index(drop=True)
    checkpoint_path = Path(checkpoint_path)
    completed = {}
    if checkpoint_path.exists():
        previous = pd.read_csv(checkpoint_path)
        for row in previous.to_dict("records"):
            if row.get("roi_path") and Path(str(row["roi_path"])).exists():
                completed[str(row["image_name"])] = row

    pending = frame.loc[~frame.image_name.astype(str).isin(completed)].copy()
    pending_lookup = pending.set_index("original_path").to_dict(orient="index")
    print(f"ROI generation: {len(completed)} reusable; {len(pending)} pending.")
    new_records = []
    inference_dataset = make_segmentation_inference_dataset(pending.original_path.astype(str).tolist())
    processed = 0

    for path_batch, image_batch in tqdm(inference_dataset, desc="ResUNet ROI generation"):
        predicted_probabilities = segmenter.predict_on_batch(image_batch).astype(np.float32)[..., 0]
        for encoded_path, small_probability in zip(path_batch.numpy(), predicted_probabilities):
            original_path = encoded_path.decode("utf-8")
            source_row = pending_lookup[original_path]
            original = load_rgb_pil(original_path)
            probability_at_original_size = np.asarray(
                Image.fromarray(small_probability.astype(np.float32), mode="F").resize(
                    original.size, Image.Resampling.BILINEAR
                ), dtype=np.float32
            )
            binary_mask = prepare_mask_for_roi(probability_at_original_size)
            roi, bounding_box, fallback_used, fallback_reason = extract_lesion_roi(original, binary_mask)
            roi_path = ROI_DIR / f"{source_row['image_name']}.jpg"
            Image.fromarray(roi).save(roi_path, format="JPEG", quality=95, optimize=True)
            record = dict(source_row)
            record.update({
                "original_path": original_path,
                "roi_path": str(roi_path),
                "bbox": json.dumps([int(value) for value in bounding_box]),
                "roi_fallback_used": bool(fallback_used),
                "roi_fallback_reason": fallback_reason,
            })
            new_records.append(record)
        processed += len(path_batch)
        if processed % 500 < SEG_BATCH_SIZE:
            pd.DataFrame(list(completed.values()) + new_records).to_csv(checkpoint_path, index=False)

    result = pd.DataFrame(list(completed.values()) + new_records)
    if len(result) != len(frame) or result.image_name.nunique() != len(frame):
        raise RuntimeError("ROI manifest is incomplete or contains duplicate image IDs.")
    if not result.roi_path.map(lambda value: Path(value).exists()).all():
        raise RuntimeError("At least one generated ROI path is missing.")
    result = frame[["image_name"]].merge(result, on="image_name", validate="one_to_one")
    result.to_csv(checkpoint_path, index=False)
    return result


classification_for_roi = pd.concat(
    [classification_train, classification_validation, classification_test], ignore_index=True
).drop_duplicates("image_name")
classification_roi_manifest = generate_roi_manifest(
    classification_for_roi, selected_segmenter, OUTPUT_DIR / "classification_roi_manifest.csv"
)

def attach_roi_data(split_frame):
    roi_columns = ["image_name", "roi_path", "bbox", "roi_fallback_used", "roi_fallback_reason"]
    return split_frame.merge(classification_roi_manifest[roi_columns], on="image_name", validate="one_to_one")


classification_train = attach_roi_data(classification_train)
classification_validation = attach_roi_data(classification_validation)
classification_test = attach_roi_data(classification_test)
for split_name, split_frame in [
    ("train", classification_train),
    ("validation", classification_validation),
    ("test", classification_test),
]:
    if split_frame.roi_path.isna().any() or not split_frame.roi_path.map(lambda value: Path(value).exists()).all():
        raise RuntimeError(f"{split_name} contains missing ROI files.")
    split_frame.to_csv(OUTPUT_DIR / f"classification_{split_name}_split.csv", index=False)

roi_manifest_columns = [
    column for column in [
        "image_id", "image_name", "patient_id", "lesion_id", "original_path", "roi_path",
        "target", "bbox", "roi_fallback_used", "roi_fallback_reason",
    ] if column in classification_roi_manifest.columns
]
classification_roi_manifest[roi_manifest_columns].to_csv(
    OUTPUT_DIR / "classification_roi_manifest.csv", index=False
)
print("ROI fallback rate:", classification_roi_manifest.roi_fallback_used.astype(bool).mean())
display(classification_roi_manifest[roi_manifest_columns].head())


## 26. tf.data Classification Pipeline


In [ ]:
classification_augmenter = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    layers.RandomRotation(0.06, fill_mode="reflect", seed=SEED + 1),
    layers.RandomZoom(
        height_factor=(-0.08, 0.08), width_factor=(-0.08, 0.08),
        fill_mode="reflect", seed=SEED + 2,
    ),
    layers.RandomTranslation(0.05, 0.05, fill_mode="reflect", seed=SEED + 3),
    layers.RandomContrast(0.10, seed=SEED + 4),
], name="classification_augmentation")


def decode_classification_image(path, target, training=False):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, CLS_IMAGE_SIZE, method="bilinear")
    image = tf.cast(image, tf.float32)
    if training:
        image = classification_augmenter(image, training=True)
        image = tf.clip_by_value(image, 0.0, 255.0)
    image.set_shape((*CLS_IMAGE_SIZE, 3))
    return image, tf.cast(target, tf.float32)


def make_classification_dataset(frame, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((
        frame.roi_path.astype(str).to_numpy(),
        frame.target.astype(np.float32).to_numpy(),
    ))
    if training:
        dataset = dataset.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(
        lambda path, target: decode_classification_image(path, target, training=training),
        num_parallel_calls=AUTOTUNE,
        deterministic=True,
    )
    return dataset.batch(CLS_BATCH_SIZE, drop_remainder=False).prefetch(AUTOTUNE)


train_cls_ds = make_classification_dataset(classification_train, training=True)
val_cls_ds = make_classification_dataset(classification_validation, training=False)
test_cls_ds = make_classification_dataset(classification_test, training=False)

sample_images, sample_targets = next(iter(val_cls_ds))
print("Classifier input shape:", sample_images.shape)
print("Pixel range:", float(tf.reduce_min(sample_images)), "to", float(tf.reduce_max(sample_images)))
assert float(tf.reduce_min(sample_images)) >= 0.0 and float(tf.reduce_max(sample_images)) <= 255.0


## 27. Class Imbalance


In [ ]:
training_class_counts = classification_train.target.value_counts().sort_index()
if set(training_class_counts.index) != {0, 1}:
    raise RuntimeError("Both classes must be present in the classification training split.")
class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=classification_train.target.to_numpy(),
)
CLASS_WEIGHTS = {0: float(class_weight_values[0]), 1: float(class_weight_values[1])}
print("Training-only class counts:", training_class_counts.to_dict())
print("Training-only class weights:", CLASS_WEIGHTS)


## 28. EfficientNetB0


Keras EfficientNetB0 includes its own input rescaling and expects float pixel values in the 0–255 range. Therefore the `tf.data` classification decoder does **not** divide by 255 and no separate `preprocess_input` call is applied. This prevents double normalisation.


In [ ]:
def build_efficientnet_classifier(input_shape=(*CLS_IMAGE_SIZE, 3), weights="imagenet"):
    inputs = keras.Input(shape=input_shape, dtype=tf.float32, name="lesion_roi")
    backbone = keras.applications.EfficientNetB0(
        include_top=False,
        weights=weights,
        input_shape=input_shape,
    )
    backbone.trainable = False
    features = backbone(inputs, training=False)
    pooled = layers.GlobalAveragePooling2D(name="classification_gap")(features)
    dropped = layers.Dropout(0.30, name="classification_dropout")(pooled)
    outputs = layers.Dense(
        1, activation="sigmoid", dtype="float32", name="malignancy_probability"
    )(dropped)
    return keras.Model(inputs, outputs, name="EfficientNetB0_Melanoma"), backbone


def classifier_metrics():
    return [
        keras.metrics.BinaryAccuracy(name="binary_accuracy"),
        keras.metrics.AUC(curve="ROC", name="roc_auc"),
        keras.metrics.AUC(curve="PR", name="pr_auc"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
    ]


def compile_classifier(model, learning_rate):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=classifier_metrics(),
    )
    return model


classifier_preview, preview_backbone = build_efficientnet_classifier(weights=None)
preview_output = classifier_preview(tf.zeros((1, *CLS_IMAGE_SIZE, 3), dtype=tf.float32), training=False)
assert tuple(preview_output.shape) == (1, 1)
print("TensorFlow version:", tf.__version__)
print("EfficientNetB0 synthetic output:", preview_output.shape)
print("EfficientNetB0 classifier parameters:", f"{classifier_preview.count_params():,}")
print("Internal Rescaling layers:", [layer.name for layer in preview_backbone.layers if isinstance(layer, layers.Rescaling)])
del classifier_preview, preview_backbone, preview_output
gc.collect()


## 29. Head Training


In [ ]:
def classifier_callbacks(checkpoint_path, patience=5):
    return [
        keras.callbacks.ModelCheckpoint(
            checkpoint_path, monitor="val_roc_auc", mode="max", save_best_only=True, verbose=1
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_roc_auc", mode="max", patience=patience, restore_best_weights=False, verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_roc_auc", mode="max", factor=0.3, patience=3, min_lr=1e-7, verbose=1
        ),
    ]


EFFICIENTNET_HEAD_PATH = OUTPUT_DIR / "efficientnet_b0_head_best.keras"
efficientnet_model, efficientnet_backbone = build_efficientnet_classifier(weights="imagenet")
efficientnet_model = compile_classifier(efficientnet_model, CLS_HEAD_LEARNING_RATE)
head_training_started = time.perf_counter()
head_history_object = efficientnet_model.fit(
    train_cls_ds,
    validation_data=val_cls_ds,
    epochs=CLS_HEAD_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=classifier_callbacks(EFFICIENTNET_HEAD_PATH),
    verbose=1,
)
HEAD_TRAINING_SECONDS = time.perf_counter() - head_training_started
head_history = pd.DataFrame(head_history_object.history)
head_history.index = np.arange(1, len(head_history) + 1)
head_history.index.name = "epoch"
head_history.to_csv(OUTPUT_DIR / "efficientnet_b0_head_history.csv")
print(f"Head-training time: {HEAD_TRAINING_SECONDS / 60:.2f} minutes")


## 30. Fine-Tuning


In [ ]:
def unfreeze_efficientnet_top(model, last_n=CLS_FINE_TUNE_LAST_N):
    backbone = model.get_layer("efficientnetb0")
    backbone.trainable = True
    cutoff = max(0, len(backbone.layers) - int(last_n))
    for layer_index, layer in enumerate(backbone.layers):
        layer.trainable = layer_index >= cutoff and not isinstance(layer, layers.BatchNormalization)
    print("Trainable upper backbone layers:", sum(layer.trainable for layer in backbone.layers))
    print("All BatchNormalization layers remain frozen:", all(
        not layer.trainable for layer in backbone.layers if isinstance(layer, layers.BatchNormalization)
    ))
    return model


def predict_probabilities(model, frame):
    dataset = make_classification_dataset(frame, training=False)
    probabilities = model.predict(dataset, verbose=1).reshape(-1).astype(float)
    if len(probabilities) != len(frame):
        raise RuntimeError("Prediction count does not match the classification manifest.")
    return probabilities


EFFICIENTNET_FINE_PATH = OUTPUT_DIR / "efficientnet_b0_finetuned_best.keras"
fine_history = pd.DataFrame()
FINE_TUNING_SECONDS = 0.0

if RUN_FINE_TUNING:
    fine_tune_model = keras.models.load_model(EFFICIENTNET_HEAD_PATH, compile=False)
    fine_tune_model = unfreeze_efficientnet_top(fine_tune_model)
    fine_tune_model = compile_classifier(fine_tune_model, CLS_FINE_TUNE_LEARNING_RATE)
    fine_tuning_started = time.perf_counter()
    fine_history_object = fine_tune_model.fit(
        train_cls_ds,
        validation_data=val_cls_ds,
        epochs=CLS_FINE_TUNE_EPOCHS,
        class_weight=CLASS_WEIGHTS,
        callbacks=classifier_callbacks(EFFICIENTNET_FINE_PATH),
        verbose=1,
    )
    FINE_TUNING_SECONDS = time.perf_counter() - fine_tuning_started
    fine_history = pd.DataFrame(fine_history_object.history)
    fine_history.index = np.arange(1, len(fine_history) + 1)
    fine_history.index.name = "epoch"
    fine_history.to_csv(OUTPUT_DIR / "efficientnet_b0_fine_tune_history.csv")

candidate_checkpoints = [("frozen_head", EFFICIENTNET_HEAD_PATH)]
if RUN_FINE_TUNING and EFFICIENTNET_FINE_PATH.exists():
    candidate_checkpoints.append(("fine_tuned", EFFICIENTNET_FINE_PATH))

validation_checkpoint_rows = []
for phase_name, checkpoint_path in candidate_checkpoints:
    candidate_model = keras.models.load_model(checkpoint_path, compile=False)
    candidate_probabilities = predict_probabilities(candidate_model, classification_validation)
    validation_checkpoint_rows.append({
        "phase": phase_name,
        "path": str(checkpoint_path),
        "validation_roc_auc": roc_auc_score(classification_validation.target, candidate_probabilities),
        "validation_pr_auc": average_precision_score(classification_validation.target, candidate_probabilities),
    })
    del candidate_model
    gc.collect()

validation_checkpoint_table = pd.DataFrame(validation_checkpoint_rows).sort_values(
    ["validation_roc_auc", "validation_pr_auc"], ascending=False
).reset_index(drop=True)
display(validation_checkpoint_table)

SELECTED_EFFICIENTNET_SOURCE = Path(validation_checkpoint_table.loc[0, "path"])
EFFICIENTNET_BEST_PATH = OUTPUT_DIR / "efficientnet_b0_best.keras"
SELECTED_CLASSIFIER_PATH = OUTPUT_DIR / "selected_classifier.keras"
shutil.copy2(SELECTED_EFFICIENTNET_SOURCE, EFFICIENTNET_BEST_PATH)
shutil.copy2(SELECTED_EFFICIENTNET_SOURCE, SELECTED_CLASSIFIER_PATH)
final_classifier = keras.models.load_model(SELECTED_CLASSIFIER_PATH, compile=False)
validation_checkpoint_table.to_csv(OUTPUT_DIR / "efficientnet_validation_checkpoint_comparison.csv", index=False)
print("Selected using validation data only:", validation_checkpoint_table.loc[0, "phase"])

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for axis, metric, title in [
    (axes[0], "loss", "Classification Loss"),
    (axes[1], "roc_auc", "Classification ROC-AUC"),
    (axes[2], "pr_auc", "Classification PR-AUC"),
]:
    head_epochs = np.arange(1, len(head_history) + 1)
    axis.plot(head_epochs, head_history[metric], label="Head training")
    axis.plot(head_epochs, head_history[f"val_{metric}"], label="Head validation")
    if len(fine_history):
        fine_epochs = np.arange(len(head_history) + 1, len(head_history) + len(fine_history) + 1)
        axis.plot(fine_epochs, fine_history[metric], label="Fine-tuning")
        axis.plot(fine_epochs, fine_history[f"val_{metric}"], label="Fine-tune validation")
        axis.axvline(len(head_history) + 0.5, color="black", linestyle="--", alpha=0.4)
    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.grid(alpha=0.25)
    axis.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "efficientnet_training_curves.png", dpi=180, bbox_inches="tight")
plt.show()


## 31. Optional MobileNetV2 Baseline


In [ ]:
def build_mobilenetv2_baseline(input_shape=(*CLS_IMAGE_SIZE, 3), weights="imagenet"):
    inputs = keras.Input(shape=input_shape, dtype=tf.float32, name="lesion_roi")
    scaled = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="mobilenet_preprocessing")(inputs)
    backbone = keras.applications.MobileNetV2(include_top=False, weights=weights, input_shape=input_shape)
    backbone.trainable = False
    features = backbone(scaled, training=False)
    pooled = layers.GlobalAveragePooling2D(name="baseline_gap")(features)
    dropped = layers.Dropout(0.30, name="baseline_dropout")(pooled)
    outputs = layers.Dense(1, activation="sigmoid", dtype="float32", name="malignancy_probability")(dropped)
    return keras.Model(inputs, outputs, name="MobileNetV2_Baseline")


# The architecture is always instantiated and tested; training remains optional.
baseline_preview = build_mobilenetv2_baseline(weights=None)
baseline_preview_output = baseline_preview(tf.zeros((1, *CLS_IMAGE_SIZE, 3)), training=False)
assert tuple(baseline_preview_output.shape) == (1, 1)
print("MobileNetV2 baseline synthetic forward pass passed.")
del baseline_preview, baseline_preview_output
gc.collect()

if RUN_CLASSIFIER_BASELINE:
    baseline_path = OUTPUT_DIR / "mobilenetv2_baseline_best.keras"
    baseline_model = compile_classifier(build_mobilenetv2_baseline(weights="imagenet"), CLS_HEAD_LEARNING_RATE)
    baseline_history_object = baseline_model.fit(
        train_cls_ds,
        validation_data=val_cls_ds,
        epochs=CLS_HEAD_EPOCHS,
        class_weight=CLASS_WEIGHTS,
        callbacks=classifier_callbacks(baseline_path),
        verbose=1,
    )
    baseline_model = keras.models.load_model(baseline_path, compile=False)
    baseline_validation_probabilities = predict_probabilities(baseline_model, classification_validation)
    baseline_test_probabilities = predict_probabilities(baseline_model, classification_test)
    baseline_metrics_table = pd.DataFrame([{
        "model": "MobileNetV2",
        "validation_roc_auc": roc_auc_score(classification_validation.target, baseline_validation_probabilities),
        "validation_pr_auc": average_precision_score(classification_validation.target, baseline_validation_probabilities),
        "test_roc_auc": roc_auc_score(classification_test.target, baseline_test_probabilities),
        "test_pr_auc": average_precision_score(classification_test.target, baseline_test_probabilities),
    }])
    baseline_metrics_table.to_csv(OUTPUT_DIR / "mobilenetv2_baseline_metrics.csv", index=False)
    display(baseline_metrics_table)
else:
    print("Baseline training disabled. Set RUN_CLASSIFIER_BASELINE=True in Section 2 to enable it.")


## 32. Threshold Selection


In [ ]:
validation_probabilities = predict_probabilities(final_classifier, classification_validation)
if OPTIMISE_THRESHOLD_ON_VALIDATION:
    validation_fpr, validation_tpr, validation_thresholds = roc_curve(
        classification_validation.target, validation_probabilities
    )
    finite_thresholds = np.isfinite(validation_thresholds)
    youden_j = validation_tpr[finite_thresholds] - validation_fpr[finite_thresholds]
    selected_index = int(np.argmax(youden_j))
    SELECTED_CLS_THRESHOLD = float(np.clip(validation_thresholds[finite_thresholds][selected_index], 0.0, 1.0))
    threshold_method = "maximum Youden J on validation predictions only"
else:
    SELECTED_CLS_THRESHOLD = DEFAULT_CLS_THRESHOLD
    threshold_method = "fixed 0.5"

threshold_payload = {
    "default_threshold": DEFAULT_CLS_THRESHOLD,
    "selected_threshold": SELECTED_CLS_THRESHOLD,
    "selection_data": "validation_only",
    "method": threshold_method,
    "segmentation_threshold": SEG_THRESHOLD,
}
(OUTPUT_DIR / "classification_threshold.json").write_text(
    json.dumps(threshold_payload, indent=2), encoding="utf-8"
)
print(json.dumps(threshold_payload, indent=2))


## 33. Final Classification Metrics


In [ ]:
def classification_metrics_at_threshold(y_true, probabilities, threshold):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        "threshold": float(threshold),
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "sensitivity": float(tp / (tp + fn)) if tp + fn else np.nan,
        "specificity": float(tn / (tn + fp)) if tn + fp else np.nan,
        "accuracy": float(accuracy_score(y_true, predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, predictions)),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "f1_score": float(f1_score(y_true, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, probabilities)),
        "pr_auc": float(average_precision_score(y_true, probabilities)),
    }


test_probabilities = predict_probabilities(final_classifier, classification_test)
classification_metrics_table = pd.DataFrame([
    {"evaluation": "threshold_0.5", **classification_metrics_at_threshold(
        classification_test.target, test_probabilities, DEFAULT_CLS_THRESHOLD
    )},
    {"evaluation": "validation_selected_threshold", **classification_metrics_at_threshold(
        classification_test.target, test_probabilities, SELECTED_CLS_THRESHOLD
    )},
])
classification_metrics_table.to_csv(OUTPUT_DIR / "classification_metrics.csv", index=False)
display(classification_metrics_table)

selected_test_predictions = (test_probabilities >= SELECTED_CLS_THRESHOLD).astype(int)
print("Classification report at the locked validation-selected threshold:\n")
print(classification_report(
    classification_test.target,
    selected_test_predictions,
    labels=[0, 1],
    target_names=["Benign", "Melanoma"],
    zero_division=0,
))

test_predictions = classification_test.copy()
test_predictions["malignancy_probability"] = test_probabilities
test_predictions["predicted_label"] = selected_test_predictions
test_predictions["predicted_diagnosis"] = np.where(selected_test_predictions == 1, "Melanoma class", "Benign class")
test_predictions.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)


## 34. Confusion Matrix


The matrix is arranged with **actual classes in rows** and **predicted classes in columns**. TN is a correctly predicted benign image; FP is a benign image flagged as melanoma; FN is a melanoma image predicted benign; TP is a melanoma image correctly flagged. False negatives are especially important in melanoma decision support because a missed malignancy could delay clinical assessment.


In [ ]:
selected_confusion_matrix = confusion_matrix(
    classification_test.target, selected_test_predictions, labels=[0, 1]
)
tn, fp, fn, tp = selected_confusion_matrix.ravel()

fig, axis = plt.subplots(figsize=(6.5, 5.5))
image = axis.imshow(selected_confusion_matrix, cmap="Blues")
for row in range(2):
    for column in range(2):
        label = [["TN", "FP"], ["FN", "TP"]][row][column]
        axis.text(column, row, f"{label}\n{selected_confusion_matrix[row, column]}", ha="center", va="center", fontsize=16)
axis.set_xticks([0, 1], ["Benign", "Melanoma"])
axis.set_yticks([0, 1], ["Benign", "Melanoma"])
axis.set_xlabel("Predicted")
axis.set_ylabel("Actual")
axis.set_title(f"Final Test Confusion Matrix\nThreshold = {SELECTED_CLS_THRESHOLD:.3f}")
fig.colorbar(image, ax=axis, fraction=0.046)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "classification_confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()
print({"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)})


## 35. ROC / PR Curves


ROC-AUC and PR-AUC use continuous malignancy probabilities, not thresholded predictions. Precision–recall analysis is particularly informative here because melanoma prevalence is low and accuracy can be dominated by the benign class.


In [ ]:
test_fpr, test_tpr, _ = roc_curve(classification_test.target, test_probabilities)
precision_curve, recall_curve, _ = precision_recall_curve(classification_test.target, test_probabilities)
test_roc_auc = roc_auc_score(classification_test.target, test_probabilities)
test_pr_auc = average_precision_score(classification_test.target, test_probabilities)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(test_fpr, test_tpr, label=f"ROC-AUC = {test_roc_auc:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="grey")
axes[0].set(xlabel="False Positive Rate", ylabel="True Positive Rate / Sensitivity", title="Final Test ROC Curve")
axes[1].plot(recall_curve, precision_curve, label=f"PR-AUC = {test_pr_auc:.3f}")
axes[1].axhline(classification_test.target.mean(), linestyle="--", color="grey", label="Prevalence baseline")
axes[1].set(xlabel="Recall / Sensitivity", ylabel="Precision", title="Final Test Precision–Recall Curve")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "classification_roc_pr_curves.png", dpi=180, bbox_inches="tight")
plt.show()


## 36. TP/TN/FP/FN Analysis


In [ ]:
test_predictions["outcome"] = np.select(
    [
        (test_predictions.target == 1) & (test_predictions.predicted_label == 1),
        (test_predictions.target == 0) & (test_predictions.predicted_label == 0),
        (test_predictions.target == 0) & (test_predictions.predicted_label == 1),
        (test_predictions.target == 1) & (test_predictions.predicted_label == 0),
    ],
    ["TP", "TN", "FP", "FN"],
    default="unknown",
)
test_predictions.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
outcome_summary = test_predictions.outcome.value_counts().reindex(["TP", "TN", "FP", "FN"], fill_value=0)
display(outcome_summary.rename("count").to_frame())

false_negative_table = test_predictions.query("outcome == 'FN'").sort_values("malignancy_probability")
print("False negatives:", len(false_negative_table))
if len(false_negative_table):
    display(false_negative_table[["image_name", "patient_id", "target", "malignancy_probability", "predicted_diagnosis"]].head(20))
else:
    print("No false negatives occurred at the locked threshold in this run.")


## 37. Grad-CAM


Grad-CAM identifies spatial areas in the lesion ROI that strongly influenced the selected network score. It does **not** demonstrate clinical causality, prove that highlighted structures are diagnostic, or replace expert interpretation.


In [ ]:
def get_efficientnet_backbone(classifier):
    for layer in classifier.layers:
        if isinstance(layer, keras.Model) and "efficientnet" in layer.name.lower():
            return layer
    raise ValueError("EfficientNet backbone was not found inside the classifier.")


def find_last_spatial_layer(backbone):
    for layer in reversed(backbone.layers):
        try:
            if len(tuple(layer.output.shape)) == 4:
                return layer
        except Exception:
            continue
    raise ValueError("No four-dimensional spatial layer was found.")


def make_gradcam_heatmap(classifier, image_batch, target_class=1):
    backbone = get_efficientnet_backbone(classifier)
    last_spatial_layer = find_last_spatial_layer(backbone)
    feature_model = keras.Model(backbone.input, last_spatial_layer.output)
    image_batch = tf.cast(image_batch, tf.float32)
    with tf.GradientTape() as tape:
        feature_maps = feature_model(image_batch, training=False)
        tape.watch(feature_maps)
        pooled = classifier.get_layer("classification_gap")(feature_maps)
        dropped = classifier.get_layer("classification_dropout")(pooled, training=False)
        malignant_probability = classifier.get_layer("malignancy_probability")(dropped)[:, 0]
        target_score = malignant_probability if int(target_class) == 1 else 1.0 - malignant_probability
    gradients = tape.gradient(target_score, feature_maps)
    if gradients is None:
        raise RuntimeError("Grad-CAM gradients could not be calculated.")
    weights = tf.reduce_mean(gradients, axis=(1, 2), keepdims=True)
    heatmap = tf.nn.relu(tf.reduce_sum(weights * feature_maps, axis=-1))[0]
    heatmap = heatmap / tf.maximum(tf.reduce_max(heatmap), EPSILON)
    return heatmap.numpy().astype(np.float32), last_spatial_layer.name


def heatmap_overlay(image, heatmap, alpha=0.40):
    image_array = np.asarray(image.convert("RGB") if isinstance(image, Image.Image) else image).astype(np.uint8)
    resized_heatmap = np.asarray(
        Image.fromarray(np.asarray(heatmap, dtype=np.float32), mode="F").resize(
            (image_array.shape[1], image_array.shape[0]), Image.Resampling.BILINEAR
        ), dtype=np.float32
    )
    coloured = (plt.get_cmap("jet")(np.clip(resized_heatmap, 0.0, 1.0))[..., :3] * 255).astype(np.uint8)
    return np.clip((1.0 - alpha) * image_array + alpha * coloured, 0, 255).astype(np.uint8)


def segmentation_overlay(image, binary_mask, alpha=0.35):
    image_array = np.asarray(image.convert("RGB") if isinstance(image, Image.Image) else image).astype(np.uint8)
    mask = np.asarray(binary_mask).squeeze().astype(bool)
    if mask.shape != image_array.shape[:2]:
        mask = np.asarray(
            Image.fromarray(mask.astype(np.uint8) * 255).resize(
                (image_array.shape[1], image_array.shape[0]), Image.Resampling.NEAREST
            )
        ) > 127
    red = np.zeros_like(image_array)
    red[..., 0] = 255
    return np.where(mask[..., None], np.clip((1 - alpha) * image_array + alpha * red, 0, 255), image_array).astype(np.uint8)


def explain_classification_row(row):
    original = load_rgb_pil(row.original_path)
    segmentation_probability = predict_resized_segmentation(selected_segmenter, original)
    binary_small = prepare_mask_for_roi(segmentation_probability)
    binary_original = np.asarray(
        Image.fromarray(binary_small * 255).resize(original.size, Image.Resampling.NEAREST)
    ) > 127
    roi_array, bbox, fallback, reason = extract_lesion_roi(original, binary_original)
    roi_image = Image.fromarray(roi_array)
    classifier_image = roi_image.resize((CLS_IMAGE_SIZE[1], CLS_IMAGE_SIZE[0]), Image.Resampling.BILINEAR)
    classifier_batch = np.asarray(classifier_image, dtype=np.float32)[None, ...]
    probability = float(final_classifier.predict(classifier_batch, verbose=0)[0, 0])
    predicted_class = int(probability >= SELECTED_CLS_THRESHOLD)
    heatmap, layer_name = make_gradcam_heatmap(final_classifier, classifier_batch, predicted_class)
    return {
        "original": np.asarray(original),
        "mask": binary_original.astype(np.uint8),
        "segmentation_overlay": segmentation_overlay(original, binary_original),
        "roi": roi_array,
        "gradcam_overlay": heatmap_overlay(roi_image, heatmap),
        "probability": probability,
        "predicted_class": predicted_class,
        "bbox": bbox,
        "fallback": fallback,
        "fallback_reason": reason,
        "gradcam_layer": layer_name,
    }


def display_explained_case(row):
    result = explain_classification_row(row)
    fig, axes = plt.subplots(1, 5, figsize=(18, 3.7))
    for axis, image_data, title, cmap in [
        (axes[0], result["original"], "Original dermoscopic image", None),
        (axes[1], result["mask"], "ResUNet mask", "gray"),
        (axes[2], result["segmentation_overlay"], "Segmentation overlay", None),
        (axes[3], result["roi"], "Lesion ROI", None),
        (axes[4], result["gradcam_overlay"], "Grad-CAM overlay", None),
    ]:
        axis.imshow(image_data, cmap=cmap)
        axis.set_title(title)
        axis.axis("off")
    true_name = "Melanoma" if int(row.target) == 1 else "Benign"
    predicted_name = "Melanoma class" if result["predicted_class"] else "Benign class"
    fig.suptitle(
        f"{row.outcome} | True: {true_name} | Predicted: {predicted_name} | "
        f"Model probability of malignancy: {result['probability']:.3f}", y=1.04
    )
    plt.tight_layout()
    plt.show()


for outcome in ["TP", "TN", "FP", "FN"]:
    available = test_predictions.query("outcome == @outcome")
    if len(available):
        display_explained_case(available.iloc[0])
    else:
        print(f"No {outcome} example is available in this run.")

if len(false_negative_table):
    print("Dedicated false-negative Grad-CAM review")
    for _, false_negative_row in false_negative_table.head(5).iterrows():
        display_explained_case(false_negative_row)


## 38. Skin-Tone Bias and Limitations


### Skin-Tone Diversity and Fairness Limitation

The ISIC datasets used here do not provide sufficiently reliable and balanced skin-tone annotations for a scientifically valid subgroup fairness analysis. This project therefore cannot honestly claim equivalent performance across skin tones. It does not invent Fitzpatrick labels, use mean image brightness as a skin-tone proxy, infer ethnicity, or artificially darken images and claim that bias has been solved.

Future work requires more diverse dermoscopic datasets, expert skin-tone annotation, subgroup sensitivity and specificity analysis, external validation, and reweighting or oversampling only when genuine subgroup labels exist.

### Other limitations

- SIIM-ISIC 2020 is highly imbalanced, even after class weighting.
- Domain shift can occur across clinics, cameras and acquisition protocols; dermoscopic images are not equivalent to ordinary smartphone photographs.
- Hair, ruler markings, gel bubbles, reflections, low contrast and lighting differences can affect segmentation and classification.
- Segmentation errors can propagate into ROI cropping and classification.
- False negatives remain possible and are particularly consequential for melanoma decision support.
- Grad-CAM is a coarse influence map, not a causal or histopathological explanation.
- Clinical use would require prospective external validation, human-factors testing, quality systems and applicable medical-device regulatory review.


In [ ]:
def evaluate_fairness_by_group(
    predictions_dataframe,
    group_column,
    target_column="target",
    probability_column="malignancy_probability",
    threshold=SELECTED_CLS_THRESHOLD,
):
    required = {group_column, target_column, probability_column}
    missing = required - set(predictions_dataframe.columns)
    if missing:
        raise ValueError(f"Missing genuine subgroup-evaluation columns: {sorted(missing)}")
    rows = []
    for group_value, group_frame in predictions_dataframe.groupby(group_column, dropna=False):
        truth = group_frame[target_column].astype(int).to_numpy()
        probability = group_frame[probability_column].astype(float).to_numpy()
        prediction = (probability >= threshold).astype(int)
        tn_g, fp_g, fn_g, tp_g = confusion_matrix(truth, prediction, labels=[0, 1]).ravel()
        rows.append({
            "group": group_value,
            "n": len(group_frame),
            "sensitivity": tp_g / (tp_g + fn_g) if tp_g + fn_g else np.nan,
            "specificity": tn_g / (tn_g + fp_g) if tn_g + fp_g else np.nan,
            "accuracy": accuracy_score(truth, prediction),
            "f1_score": f1_score(truth, prediction, zero_division=0),
            "roc_auc": roc_auc_score(truth, probability) if len(np.unique(truth)) == 2 else np.nan,
        })
    return pd.DataFrame(rows)


print("Fairness helper defined but not run: genuine skin-tone subgroup labels are not available.")


## 39. End-to-End Prediction Function


In [ ]:
def validate_input_image(image):
    if image is None:
        raise ValueError("No dermoscopic image was supplied.")
    if isinstance(image, (str, Path)):
        image = load_rgb_pil(image)
    elif isinstance(image, np.ndarray):
        if image.size == 0:
            raise ValueError("The supplied image array is empty.")
        image = Image.fromarray(np.asarray(image).astype(np.uint8)).convert("RGB")
    elif isinstance(image, Image.Image):
        image = image.convert("RGB")
    else:
        raise TypeError("Input must be a PIL image, NumPy array or image path.")
    if min(image.size) < 32:
        raise ValueError("Both image dimensions must be at least 32 pixels.")
    return image


def predict_melanoma(image):
    original = validate_input_image(image)
    segmentation_probability_small = predict_resized_segmentation(selected_segmenter, original)
    segmentation_probability_original = np.asarray(
        Image.fromarray(segmentation_probability_small.astype(np.float32), mode="F").resize(
            original.size, Image.Resampling.BILINEAR
        ), dtype=np.float32
    )
    predicted_mask = prepare_mask_for_roi(segmentation_probability_original)
    roi_array, bounding_box, fallback_used, fallback_reason = extract_lesion_roi(original, predicted_mask)
    roi_image = Image.fromarray(roi_array)
    classifier_image = roi_image.resize((CLS_IMAGE_SIZE[1], CLS_IMAGE_SIZE[0]), Image.Resampling.BILINEAR)
    classifier_batch = np.asarray(classifier_image, dtype=np.float32)[None, ...]
    malignancy_probability = float(final_classifier.predict(classifier_batch, verbose=0)[0, 0])
    predicted_label = int(malignancy_probability >= SELECTED_CLS_THRESHOLD)
    predicted_diagnosis = "Melanoma class" if predicted_label else "Benign class"
    heatmap, gradcam_layer = make_gradcam_heatmap(final_classifier, classifier_batch, predicted_label)

    return {
        "original_image": np.asarray(original),
        "predicted_mask": predicted_mask.astype(np.uint8),
        "segmentation_overlay": segmentation_overlay(original, predicted_mask),
        "roi": roi_array,
        "bounding_box": tuple(int(value) for value in bounding_box),
        "roi_fallback_used": bool(fallback_used),
        "roi_fallback_reason": fallback_reason,
        "malignancy_probability": malignancy_probability,
        "predicted_label": predicted_label,
        "predicted_diagnosis": predicted_diagnosis,
        "classification_threshold": float(SELECTED_CLS_THRESHOLD),
        "gradcam_overlay": heatmap_overlay(roi_image, heatmap),
        "gradcam_layer": gradcam_layer,
    }


inference_smoke_result = predict_melanoma(classification_test.iloc[0].original_path)
required_prediction_keys = {
    "original_image", "predicted_mask", "segmentation_overlay", "roi", "bounding_box",
    "roi_fallback_used", "malignancy_probability", "predicted_label",
    "predicted_diagnosis", "gradcam_overlay",
}
assert required_prediction_keys.issubset(inference_smoke_result)
print("End-to-end real-image inference test passed.")


## 40. Gradio Web Application


In [ ]:
import gradio as gr


DISCLAIMER = (
    "Research/educational prototype only. This system is not a medical diagnosis and must not "
    "replace assessment by a qualified healthcare professional."
)


def gradio_predict(image):
    result = predict_melanoma(image)
    probability_text = f"{100.0 * result['malignancy_probability']:.2f}%"
    thresholded_output = (
        f"Model output: {result['predicted_diagnosis']} at the locked validation threshold "
        f"({result['classification_threshold']:.3f})."
    )
    return (
        result["original_image"],
        result["predicted_mask"] * 255,
        result["segmentation_overlay"],
        result["roi"],
        result["gradcam_overlay"],
        probability_text,
        thresholded_output,
    )


with gr.Blocks(title="AI-Assisted Melanoma Research Prototype") as demo:
    gr.Markdown("# AI-Assisted Dermoscopic Melanoma Analysis")
    gr.Markdown(f"**{DISCLAIMER}**")
    input_image = gr.Image(type="pil", label="Upload a dermoscopic image")
    analyse_button = gr.Button("Analyse image", variant="primary")
    with gr.Row():
        original_output = gr.Image(label="1. Original image")
        mask_output = gr.Image(label="2. Predicted lesion mask")
        segmentation_output = gr.Image(label="3. Segmentation overlay")
    with gr.Row():
        roi_output = gr.Image(label="4. Lesion ROI")
        gradcam_output = gr.Image(label="5. Grad-CAM overlay")
    probability_output = gr.Textbox(label="6. Model probability of malignancy")
    prediction_output = gr.Textbox(label="7. Thresholded model output")
    analyse_button.click(
        fn=gradio_predict,
        inputs=input_image,
        outputs=[
            original_output,
            mask_output,
            segmentation_output,
            roi_output,
            gradcam_output,
            probability_output,
            prediction_output,
        ],
    )

if LAUNCH_GRADIO:
    demo.launch(share=True)
else:
    print("Gradio app is ready. Set LAUNCH_GRADIO=True in Section 2, then rerun this cell.")


## 41. Saved Outputs


In [ ]:
# Ensure the requested aliases exist in the persistent output directory.
shutil.copy2(RESUNET_BEST_PATH, OUTPUT_DIR / "selected_segmenter.keras")
shutil.copy2(EFFICIENTNET_BEST_PATH, OUTPUT_DIR / "selected_classifier.keras")

required_outputs = [
    "resunet_best.keras",
    "selected_segmenter.keras",
    "efficientnet_b0_best.keras",
    "selected_classifier.keras",
    "resunet_training_history.csv",
    "ResUNet_team_comparison_row.csv",
    "segmentation_split.csv",
    "classification_train_split.csv",
    "classification_validation_split.csv",
    "classification_test_split.csv",
    "classification_metrics.csv",
    "test_predictions.csv",
    "classification_threshold.json",
    "resunet_pipeline_config.json",
]
missing_outputs = [name for name in required_outputs if not (OUTPUT_DIR / name).exists()]
if missing_outputs:
    raise RuntimeError(f"Required persistent outputs are missing: {missing_outputs}")
print("All requested persistent outputs are present in:", OUTPUT_DIR)

# Fresh reload tests.
reloaded_segmenter = keras.models.load_model(
    OUTPUT_DIR / "resunet_best.keras", custom_objects=CUSTOM_OBJECTS, compile=False
)
segmentation_reload_output = reloaded_segmenter(
    tf.zeros((1, *SEG_IMAGE_SIZE, 3), dtype=tf.float32), training=False
)
assert tuple(segmentation_reload_output.shape) == (1, *SEG_IMAGE_SIZE, 1)

reloaded_classifier = keras.models.load_model(OUTPUT_DIR / "selected_classifier.keras", compile=False)
classification_reload_output = reloaded_classifier(
    tf.zeros((1, *CLS_IMAGE_SIZE, 3), dtype=tf.float32), training=False
)
assert tuple(classification_reload_output.shape) == (1, 1)
print("ResUNet and selected-classifier reload tests passed.")


In [ ]:
def validate_notebook_python_cells(notebook_path):
    notebook_path = Path(notebook_path)
    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    failures = []
    for cell_index, cell in enumerate(notebook.get("cells", []), start=1):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        try:
            compile(source, f"cell_{cell_index}", "exec", flags=ast.PyCF_ONLY_AST)
        except SyntaxError as exc:
            failures.append({"cell": cell_index, "error": str(exc)})
    if failures:
        raise SyntaxError(f"Notebook AST validation failures: {failures}")
    return len([cell for cell in notebook.get("cells", []) if cell.get("cell_type") == "code"])


import ast
notebook_candidates = [
    Path("/content/Melanoma_ResUNet_Full_Pipeline_Final.ipynb"),
    Path.cwd() / "Melanoma_ResUNet_Full_Pipeline_Final.ipynb",
]
mounted_notebook = next((path for path in notebook_candidates if path.exists()), None)
if mounted_notebook is not None:
    print("AST-validated code cells:", validate_notebook_python_cells(mounted_notebook))
else:
    print("Notebook source file is not mounted in this runtime; architecture, loss, ROI and reload self-tests still ran.")


## 42. References


1. ISIC Challenge. [ISIC 2018 challenge overview](https://challenge.isic-archive.com/landing/2018/) and [official challenge datasets](https://challenge.isic-archive.com/data/).
2. Codella, N. C. F., et al. (2019). [*Skin Lesion Analysis Toward Melanoma Detection 2018: A Challenge Hosted by the International Skin Imaging Collaboration (ISIC)*](https://arxiv.org/abs/1902.03368).
3. Tschandl, P., Rosendahl, C., & Kittler, H. (2018). [*The HAM10000 dataset, a large collection of multi-source dermatoscopic images of common pigmented skin lesions*](https://doi.org/10.1038/sdata.2018.161). Scientific Data, 5, 180161.
4. International Skin Imaging Collaboration. [*SIIM-ISIC 2020 Challenge Dataset*](https://doi.org/10.34970/2020-ds01) (2020).
5. Rotemberg, V., et al. (2021). [*A patient-centric dataset of images and metadata for identifying melanomas using clinical context*](https://doi.org/10.1038/s41597-021-00815-z). Scientific Data, 8, 34.
6. Ronneberger, O., Fischer, P., & Brox, T. (2015). [*U-Net: Convolutional Networks for Biomedical Image Segmentation*](https://arxiv.org/abs/1505.04597). MICCAI.
7. He, K., Zhang, X., Ren, S., & Sun, J. (2016). [*Deep Residual Learning for Image Recognition*](https://arxiv.org/abs/1512.03385). CVPR.
8. Zhang, Z., Liu, Q., & Wang, Y. (2018). [*Road Extraction by Deep Residual U-Net*](https://arxiv.org/abs/1711.10684). IEEE Geoscience and Remote Sensing Letters, 15(5), 749–753.
9. Tan, M., & Le, Q. V. (2019). [*EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks*](https://proceedings.mlr.press/v97/tan19a.html). ICML.
10. Selvaraju, R. R., et al. (2017). [*Grad-CAM: Visual Explanations from Deep Networks via Gradient-Based Localization*](https://openaccess.thecvf.com/content_iccv_2017/html/Selvaraju_Grad-CAM_Visual_Explanations_ICCV_2017_paper.html). ICCV.

### Reporting note

Report only metrics produced by a completed full run. Retain the split CSVs, exact configuration, training histories and locked thresholds with the submitted results. Do not present this research prototype as a clinical diagnostic system.
